Fig-1

In [ ]:
# First to fifth mode weights W1-W5
# Plotting time range: 2018-09-06 to 2020-05-06
'''
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import AutoMinorLocator
from matplotlib.lines import Line2D


# ========================
# Global plotting parameters: double-column paper style
# ========================
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['mathtext.fontset'] = 'stix'
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['xtick.labelsize'] = 11
plt.rcParams['ytick.labelsize'] = 11
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['axes.linewidth'] = 0.7
plt.rcParams['xtick.major.width'] = 0.7
plt.rcParams['ytick.major.width'] = 0.7
plt.rcParams['xtick.minor.width'] = 0.5
plt.rcParams['ytick.minor.width'] = 0.5


# ========================
# 1. Read Excel data
# ========================
def load_eigenvalue_data(excel_path):
    df = pd.read_excel(excel_path)
    df['date'] = pd.to_datetime(df['date'])

    required_cols = [
        'date',
        'lambda_1_102',
        'lambda_2_102',
        'lambda_3_102',
        'lambda_4_102',
        'lambda_5_102'
    ]

    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Required column missing from the Excel file: {col}")

    return df.sort_values('date').reset_index(drop=True)


# ========================
# 2. Plot W1-W5
# ========================
def plot_w1_to_w5(df, output_dir=None):

    # Ridgecrest Mw 7.1 mainshock date
    event_date = pd.to_datetime('2019-07-06')

    # Plotting time range
    plot_start = pd.to_datetime('2018-09-06')
    plot_end = pd.to_datetime('2020-05-06')

    df_plot = df[
        (df['date'] >= plot_start) &
        (df['date'] <= plot_end)
    ].copy()

    if df_plot.empty:
        raise ValueError(
            f"No data are available within the specified time range {plot_start.date()} to "
            f"{plot_end.date()}. Please check the date column."
        )

    # ========================
    # Mode columns, labels, and colors
    # ========================
    mode_columns = [
        'lambda_1_102',
        'lambda_2_102',
        'lambda_3_102',
        'lambda_4_102',
        'lambda_5_102'
    ]

    mode_labels = [
        r'$W^1$',
        r'$W^2$',
        r'$W^3$',
        r'$W^4$',
        r'$W^5$'
    ]

    mode_colors = [
        '#1f4e79',  # W1: dark blue
        '#2e8b57',  # W2: green
        '#c23b22',  # W3: red
        '#8c6bb1',  # W4: purple
        '#d89000'   # W5: orange
    ]

    # Major x-axis ticks
    major_ticks = pd.to_datetime([
        '2018-11-01',
        '2019-03-01',
        '2019-07-01',
        '2019-11-01',
        '2020-03-01'
    ])

    # ========================
    # Create figure
    # ========================
    fig, ax = plt.subplots(
        figsize=(4.8, 3.3),
        facecolor='white'
    )

    ax.set_facecolor('white')
    ax.grid(False, which='both', axis='both')

    # ========================
    # Plot W1-W5
    # ========================
    mode_lines = []

    for column, label, color in zip(
        mode_columns,
        mode_labels,
        mode_colors
    ):
        line, = ax.plot(
            df_plot['date'],
            df_plot[column],
            color=color,
            linewidth=1.25,
            marker='o',
            markersize=1.4,
            markevery=10,
            zorder=3,
            label=label
        )

        mode_lines.append(line)

    # ========================
    # Mark the mainshock date
    # ========================
    ax.axvline(
        event_date,
        color='black',
        linestyle='--',
        linewidth=0.9,
        zorder=2
    )

    main_quake_handle = Line2D(
        [0], [0],
        color='black',
        linestyle='--',
        linewidth=0.9,
        label='Main'
    )

    # ========================
    # Axis settings
    # ========================
    ax.set_xlim(plot_start, plot_end)

    ax.set_xticks(major_ticks)
    ax.xaxis.set_major_formatter(
        mdates.DateFormatter('%Y-%m')
    )

    # Set one minor tick for each month
    ax.xaxis.set_minor_locator(
        mdates.MonthLocator(interval=1)
    )

    ax.yaxis.set_minor_locator(
        AutoMinorLocator(5)
    )

    ax.set_xlabel('Date')
    ax.set_ylabel('W')

    # Borders
    for spine in ax.spines.values():
        spine.set_linewidth(0.7)
        spine.set_color('0.2')

    # Major x-axis ticks
    ax.tick_params(
        axis='x',
        which='major',
        direction='in',
        bottom=True,
        top=False,
        labelbottom=True,
        rotation=0,
        length=4,
        width=0.7
    )

    # Minor x-axis ticks
    ax.tick_params(
        axis='x',
        which='minor',
        direction='in',
        bottom=True,
        top=False,
        length=2.5,
        width=0.5
    )

    # Major y-axis ticks
    ax.tick_params(
        axis='y',
        which='major',
        direction='in',
        left=True,
        right=False,
        length=4,
        width=0.7
    )

    # Minor y-axis ticks
    ax.tick_params(
        axis='y',
        which='minor',
        direction='in',
        left=True,
        right=False,
        length=2.5,
        width=0.5
    )

    # ========================
    # Legend
    # ========================
    ax.legend(
        handles=mode_lines + [main_quake_handle],
        loc='upper right',
        ncol=2,
        frameon=False,
        handlelength=1.8,
        columnspacing=0.8,
        handletextpad=0.4,
        borderpad=0.2
    )

    # ========================
    # Layout
    # ========================
    plt.subplots_adjust(
        left=0.15,
        right=0.97,
        top=0.97,
        bottom=0.18
    )

    # ========================
    # Save figure
    # ========================
    if output_dir is not None:
        os.makedirs(output_dir, exist_ok=True)

        png_path = os.path.join(
            output_dir,
            'W1_W5_20180906_20200506.png'
        )

        pdf_path = os.path.join(
            output_dir,
            'W1_W5_20180906_20200506.pdf'
        )

        plt.savefig(
            png_path,
            dpi=600,
            bbox_inches='tight',
            facecolor='white'
        )

        plt.savefig(
            pdf_path,
            format='pdf',
            bbox_inches='tight',
            facecolor='white'
        )

        print(
            f"Figure saved as:\n"
            f"{png_path}\n"
            f"{pdf_path}"
        )

    plt.close(fig)


# ========================
# 3. Main program
# ========================
if __name__ == "__main__":

    excel_path = (
        r"D:\a\master\Earthquake-US"
        r"\Fig-Use-2\Fig-2"
        r"\EMT_eigenvalues_entropy_data.xlsx"
    )

    output_dir = (
        r"D:\a\master\Earthquake-US"
        r"\Fig-Over-Output\Fig-S10"
    )

    df = load_eigenvalue_data(excel_path)

    plot_w1_to_w5(
        df,
        output_dir=output_dir
    )
'''

Fig-S2

In [ ]:
# (a) Eigenvalue distribution, (b) Logarithm of the ordinate
'''
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from matplotlib.ticker import AutoMinorLocator, MultipleLocator

# =========================
# Global style: Nature double-column two-panel style
# =========================
MM_TO_INCH = 1 / 25.4
FIG_WIDTH = 183 * MM_TO_INCH
FIG_HEIGHT = 80 * MM_TO_INCH   # One row and two columns; reduce the height appropriately

plt.rcParams['font.family'] = 'Arial'
plt.rcParams['mathtext.fontset'] = 'stix'
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10

plt.rcParams['axes.linewidth'] = 0.8
plt.rcParams['xtick.major.width'] = 0.8
plt.rcParams['ytick.major.width'] = 0.8
plt.rcParams['xtick.minor.width'] = 0.6
plt.rcParams['ytick.minor.width'] = 0.6

plt.rcParams['xtick.major.size'] = 4.5
plt.rcParams['ytick.major.size'] = 4.5
plt.rcParams['xtick.minor.size'] = 2.5
plt.rcParams['ytick.minor.size'] = 2.5

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42


def load_and_merge_data(csv_dir):
    """Load and merge all CSV files"""
    files = glob.glob(os.path.join(csv_dir, "*.csv"))
    if not files:
        raise FileNotFoundError(f"No CSV files found in directory: {csv_dir}")

    dfs = []
    for file in files:
        df = pd.read_csv(file)
        df['site_id'] = os.path.splitext(os.path.basename(file))[0]
        dfs.append(df)

    return pd.concat(dfs, ignore_index=True)


def preprocess_gnss_data(df, time_step=1):
    """Preprocess GNSS data (only dE and dN are processed)"""
    df = df.copy()
    df['time'] = pd.to_datetime(df['YYYYMMDD'], format='%Y%m%d')
    df = df.sort_values(['site_id', 'time']).reset_index(drop=True)

    sampled_times = np.sort(df['time'].unique())[::time_step]
    df_sampled = df[df['time'].isin(sampled_times)].copy()

    for component in ['dE', 'dN']:
        df_sampled[f'{component}_standardized'] = 0.0
        for site in df_sampled['site_id'].unique():
            site_mask = df_sampled['site_id'] == site
            scaler = StandardScaler()
            values = df_sampled.loc[site_mask, component].values.reshape(-1, 1)
            df_sampled.loc[site_mask, f'{component}_standardized'] = scaler.fit_transform(values).flatten()

    site_lengths = df_sampled.groupby('site_id').size()
    if len(site_lengths.unique()) > 1:
        print("Warning: Data lengths are inconsistent for some sites; automatic trimming will be performed")
        M = min(site_lengths)
        valid_sites = site_lengths[site_lengths == M].index
        df_sampled = df_sampled[df_sampled['site_id'].isin(valid_sites)]
    else:
        M = site_lengths.iloc[0]

    sites = sorted(df_sampled['site_id'].unique())
    N_T = 2 * len(sites)
    A = np.zeros((M, N_T))

    for i, site in enumerate(sites):
        site_data = df_sampled[df_sampled['site_id'] == site].sort_values('time').iloc[:M]
        A[:, 2 * i] = site_data['dE_standardized'].values
        A[:, 2 * i + 1] = site_data['dN_standardized'].values

    C_0 = np.sum(A ** 2)
    if C_0 <= 0:
        raise ValueError("The total energy C_0 of matrix A <= 0; normalization cannot be performed.")

    A_normalized = A / np.sqrt(C_0)
    return A_normalized, sites, sampled_times[:M]


def compute_eigen_microstates(A):
    """Calculate the eigenvalue spectrum using SVD"""
    U, S, Vt = np.linalg.svd(A.T, full_matrices=False)
    eigenvalues = S ** 2
    eigenmicrostates = U
    return eigenvalues, eigenmicrostates, Vt.T, S


def get_eigenvalue_spectra(df, target_dates, window_size=30):
    """
    Extract eigenvalue spectra for multiple target dates
    Return: original W spectra, ln(W) spectra, and labels
    """
    date_labels = [
        'Pre-seismic',
        'Co-seismic',
        'Post-seismic'
    ]

    target_dates_dt = [pd.to_datetime(date) for date in target_dates]
    all_times = pd.to_datetime(df['YYYYMMDD'], format='%Y%m%d').unique()
    all_times = np.sort(all_times)

    all_w_list = []
    all_lnw_list = []
    eps = 1e-12

    for target_date in target_dates_dt:
        time_indices = np.where(all_times <= target_date)[0]
        if len(time_indices) == 0:
            continue

        end_idx = time_indices[-1]
        start_idx = max(0, end_idx - window_size + 1)
        window_times = all_times[start_idx:end_idx + 1]

        df_window = df[df['time'].isin(window_times)].copy()
        A, sites, _ = preprocess_gnss_data(df_window)
        eigenvalues, _, _, _ = compute_eigen_microstates(A)

        vals_plot = eigenvalues[:window_size - 1]      # Use only the first 29 modes, excluding the final near-zero mode
        ln_vals = np.log(np.maximum(vals_plot, eps))   # eps protection to avoid ln(0)

        all_w_list.append(vals_plot)
        all_lnw_list.append(ln_vals)

    return all_w_list, all_lnw_list, date_labels


def style_axis(ax, show_xlabel=False):
    """Apply a unified axis style"""
    ax.set_xlim(0.3, 29.7)
    ax.xaxis.set_major_locator(MultipleLocator(4))
    ax.xaxis.set_minor_locator(MultipleLocator(1))
    ax.yaxis.set_minor_locator(AutoMinorLocator(4))

    
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.8)
        spine.set_color('black')

    ax.tick_params(
        axis='both',
        which='major',
        direction='in',
        bottom=True,
        left=True,
        top=False,
        right=False,
        length=4.5,
        width=0.8,
        labelsize=10
    )

    ax.tick_params(
        axis='both',
        which='minor',
        direction='in',
        bottom=True,
        left=True,
        top=False,
        right=False,
        length=2.5,
        width=0.6
    )

    ax.grid(False)

    if not show_xlabel:
        ax.tick_params(axis='x', labelbottom=False)
    else:
        ax.set_xlabel('Rank', fontweight='normal')


def plot_W_and_lnW_two_panel(df, target_dates, window_size=30, output_dir=None):
    """
    Left: W
    Right: ln(W)
    One row and two columns
    """
    colors = ['#ee6055', '#60d394', '#ffd97d']
    markers = ['o', 'o', 'o']
    line_styles = ['-', '-', '-']

    all_w_list, all_lnw_list, date_labels = get_eigenvalue_spectra(
        df=df,
        target_dates=target_dates,
        window_size=window_size
    )

    ranks = np.arange(1, window_size)

    fig, (ax1, ax2) = plt.subplots(
        1, 2,
        figsize=(FIG_WIDTH, FIG_HEIGHT),
        sharex=False,
        sharey=False
    )

    # =========================
    # Left panel: W
    # =========================
    handles = []
    for w_vals, label, color, marker, line_style in zip(
        all_w_list, date_labels, colors, markers, line_styles
    ):
        line, = ax1.plot(
            ranks, w_vals,
            color=color,
            marker=marker,
            linestyle=line_style,
            linewidth=1.8,
            markersize=4.8,
            markeredgewidth=0.5,
            alpha=0.95,
            label=label
        )
        handles.append(line)

    ax1.set_ylabel('W', fontweight='normal')
    style_axis(ax1, show_xlabel=True)

    ax1.set_ylim(-0.05, 0.7)
    ax1.set_yticks([0, 0.2, 0.4, 0.6])

    ax1.legend(
        handles=handles,
        loc='upper right',
        frameon=False,
        handlelength=2.2,
        borderpad=0.2,
        labelspacing=0.4
    )

    # =========================
    # Right panel: ln(W)
    # =========================
    for lnw_vals, label, color, marker, line_style in zip(
        all_lnw_list, date_labels, colors, markers, line_styles
    ):
        ax2.plot(
            ranks, lnw_vals,
            color=color,
            marker=marker,
            linestyle=line_style,
            linewidth=1.8,
            markersize=4.8,
            markeredgewidth=0.5,
            alpha=0.95
        )

    ax2.set_ylabel('ln(W)', fontweight='normal')
    style_axis(ax2, show_xlabel=True)

    ax2.set_ylim(-7, 0.5)
    ax2.set_yticks([0, -2, -4, -6])

    fig.subplots_adjust(
        left=0.08,
        right=0.98,
        bottom=0.18,
        top=0.95,
        wspace=0.2
    )

    if output_dir:
        os.makedirs(output_dir, exist_ok=True)

        png_path = os.path.join(output_dir, 'W_lnW_one_row_two_columns.png')
        pdf_path = os.path.join(output_dir, 'W_lnW_one_row_two_columns.pdf')

        plt.savefig(png_path, dpi=600, bbox_inches='tight')
        plt.savefig(pdf_path, format='pdf', bbox_inches='tight')

        print(f"Figure saved to:\n{png_path}\n{pdf_path}")

    plt.close()

    return fig, (ax1, ax2), all_w_list, all_lnw_list


# =========================
# Main program
# =========================
if __name__ == "__main__":
    output_dir = "D:/a/master/Earthquake-US/Fig-Use-2/Fig-S0/"
    csv_dir = "D:/a/master/Earthquake-US/Data/20190706-102/PosData-7"

    df_merged = load_and_merge_data(csv_dir)
    df_merged['time'] = pd.to_datetime(df_merged['YYYYMMDD'], format='%Y%m%d')

    target_dates = ['2019-05-21', '2019-07-22', '2019-09-20']

    fig, axes, w_list, lnw_list = plot_W_and_lnW_two_panel(
        df_merged,
        target_dates,
        window_size=30,
        output_dir=output_dir
    )
'''

Fig-S3 to Fig-S5

In [ ]:
# Use the image in Fig-3 for stitching and drawing

Fig-S6

In [ ]:
# (a) Plot station map for the s range
'''
import os
from pathlib import Path
import pandas as pd
import pygmt


# =========================================================
# 1. Global configuration
# =========================================================
REGION = [-118.5, -116.5, 34.8, 36.8]
PROJECTION = "M15c"

EQ_CSV = Path("D:/a/master/Earthquake-US/Fig-Use/Fig-1/ridgecrest_usgs.csv")
STATION_FOLDER = Path("D:/a/master/Earthquake-US/Data/20190706-20/PosData-7")
DEM_GRID = Path("D:/a/master/Earthquake-US/Fig-Use/Fig-1/DEM/SRTM_Ridgecrest_30m.tif")
FAULT_SHP = Path("D:/a/master/Earthquake-US/Fig-Use/Fig-1/faults/Qfaults_US_Database.shp")

OUTPUT_PNG = Path("D:/a/master/Earthquake-US/Fig-Use-2/Fig-S6/20_with_stations_gnss_major_final_v2.png")
OUTPUT_PDF = Path("D:/a/master/Earthquake-US/Fig-Use-2/Fig-S6/20_with_stations_gnss_major_final_v2.pdf")

DEPTH_RANGE = [0, 10]
MAG_MIN = 1.5
SIZE_SCALE = 0.020

PLOT_BACKGROUND_EQ = True

# -----------------------------
# Colors
# -----------------------------
MAIN_64_COLOR = "#900000"
MAIN_71_COLOR = "black"
GNSS_COLOR = "#432818"

# Mainshocks
MAINSHOCKS = [
    {
        "lon": -117.504,
        "lat": 35.705,
        "depth": 8.0,
        "style": "a0.6c",
        "fill": MAIN_64_COLOR,
        "pen": None,
        "label": "Mw 6.4 Depth 8.0 km",
    },
    {
        "lon": -117.599,
        "lat": 35.769,
        "depth": 10.5,
        "style": "a0.8c",
        "fill": MAIN_71_COLOR,
        "pen": None, 
        "label": "Mw 7.1 Depth 10.5 km",
    },
]


# =========================================================
# 2. Utilities
# =========================================================
def check_file_exists(path_obj, description="file"):
    if not path_obj.exists():
        raise FileNotFoundError(f"{description} not found: {path_obj}")


def load_earthquake_catalog(csv_path, depth_range=(0, 10), mag_min=1.5, size_scale=0.015):
    check_file_exists(csv_path, "Earthquake catalog")

    df = pd.read_csv(csv_path)

    required_cols = ["longitude", "latitude", "depth", "mag"]
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in earthquake catalog: {missing}")

    df = df[required_cols].dropna()
    df = df[
        (df["depth"] >= depth_range[0]) &
        (df["depth"] <= depth_range[1]) &
        (df["mag"] >= mag_min)
    ].copy()

    df["size"] = size_scale * df["mag"]
    return df


def load_station_positions(folder_path):
    check_file_exists(folder_path, "Station folder")

    station_lons = []
    station_lats = []

    csv_files = sorted(folder_path.glob("*.csv"))
    if not csv_files:
        print(f"No station CSV files found in: {folder_path}")
        return station_lons, station_lats

    for file_path in csv_files:
        try:
            df = pd.read_csv(file_path)

            if df.empty:
                continue

            required_cols = ["NLat", "Elong"]
            if not all(col in df.columns for col in required_cols):
                print(f"Skipped {file_path.name}: missing columns {required_cols}")
                continue

            station_lats.append(df.iloc[0]["NLat"])
            station_lons.append(df.iloc[0]["Elong"])

        except Exception as e:
            print(f"Error reading {file_path.name}: {e}")

    return station_lons, station_lats


# =========================================================
# 3. Plotting
# =========================================================
def create_map_figure(eq_df, station_lons, station_lats):
    fig = pygmt.Figure()

    # -----------------------------
    # Global style
    # -----------------------------
    pygmt.config(
        FONT="15p,Helvetica,black",
        FONT_ANNOT_PRIMARY="20p,Helvetica,black",
        FONT_LABEL="15p,Helvetica,black",
        FONT_TITLE="15p,Helvetica,black",
        MAP_FRAME_TYPE="fancy",
        MAP_FRAME_PEN="0.7p,black",
        MAP_TICK_PEN_PRIMARY="0.6p,black",
        MAP_TICK_LENGTH_PRIMARY="0.14c",
        MAP_LABEL_OFFSET="0.10c",
        MAP_ANNOT_OFFSET_PRIMARY="0.07c",
        FORMAT_GEO_MAP="dddF",
    )

    # -----------------------------
    # Background topography (paper-style shaded relief)
    # -----------------------------
    pygmt.makecpt(cmap="grayC", series=[-500, 2500], reverse=False)
    
    # Base relief
    fig.grdimage(
        grid=str(DEM_GRID),
        region=REGION,
        projection=PROJECTION,
        cmap=True,
        shading="+a315+nt0.8",
        transparency=8,
    )
    fig.coast(
    region=REGION,
    projection=PROJECTION,
    land="245/245/245@25",
    water="245/245/245@25",
    frame=False
    )

    # -----------------------------
    # Faults (weakened) 
    # -----------------------------
    fig.plot(
        data=str(FAULT_SHP),
        pen="0.12p,gray35@12"
    )

    # -----------------------------
    # CPT only for background earthquakes
    # -----------------------------
    pygmt.makecpt(cmap="hot", series=DEPTH_RANGE, reverse=True, background=True)

    # -----------------------------
    # Background earthquakes 
    # Slightly enlarged, with size still representing magnitude
    # -----------------------------
    if PLOT_BACKGROUND_EQ and len(eq_df) > 0:
        fig.plot(
            x=eq_df["longitude"],
            y=eq_df["latitude"],
            style="c",
            size=eq_df["size"] * 0.90,
            fill=eq_df["depth"],
            cmap=True,
            pen=None,
            transparency=22,
        )

    # -----------------------------
    # GNSS stations
    # -----------------------------
    if station_lons and station_lats:
        fig.plot(
            x=station_lons,
            y=station_lats,
            style="t0.34c",
            pen=f"1.05p,{GNSS_COLOR}",
        )
        print(f"Plotted {len(station_lons)} station positions")
    else:
        print("No station positions found to plot")

    # -----------------------------
    # Mainshocks
    # Fixed colors, not included in the colorbar
    # -----------------------------
    for shock in MAINSHOCKS:
        fig.plot(
            x=[shock["lon"]],
            y=[shock["lat"]],
            style=shock["style"],
            fill=shock["fill"],
            pen=shock["pen"],
        )

    # -----------------------------
    # Basemap
    # -----------------------------
    fig.basemap(
        region=REGION,
        projection=PROJECTION,
        frame=["WSen", "xa1f0.5", "ya1f0.5"]
    )

    # -----------------------------
    # Legend
    # -----------------------------
    dummy_x = REGION[0] - 10
    dummy_y = REGION[2] - 10

    fig.plot(
        x=[dummy_x],
        y=[dummy_y],
        style="t0.34c",
        pen=f"1.05p,{GNSS_COLOR}",
        label="GNSS Stations"
    )

    fig.plot(
        x=[dummy_x],
        y=[dummy_y],
        style="a0.40c",
        fill=MAIN_64_COLOR,
        pen=None,
        label="Mw 6.4 Depth 8.0 km"
    )

    fig.plot(
        x=[dummy_x],
        y=[dummy_y],
        style="a0.40c",
        fill=MAIN_71_COLOR,
        pen=None,
        label="Mw 7.1 Depth 10.5 km"
    )

    fig.legend(
        position="JTR+jTR+o0.20c",
        box=False,
    )

    # -----------------------------
    # Colorbar
    # Only for small earthquakes
    # Label above the bar
    # -----------------------------
    if PLOT_BACKGROUND_EQ and len(eq_df) > 0:
        with pygmt.config(
            FONT="13p,Helvetica,black",
            FONT_LABEL="20p,Helvetica,black",
            FONT_ANNOT_PRIMARY="13p,Helvetica,black"
        ):
            fig.colorbar(
                cmap=True,
                frame=["x2+lDepth\\040(km)"],
                position="jTR+o1.25c/2.65c+w5.2c/0.42c+v",
                box=False
            )

    # -----------------------------
    # Scale bar
    # -----------------------------
    with pygmt.config(
        FONT_ANNOT_PRIMARY="11p,Helvetica,black",
        FONT_LABEL="12p,Helvetica,black"
    ):
        fig.basemap(
            map_scale="jTR+w20k+o2.25c/3.65c+f+lkm"
        )

    # -----------------------------
    # Inset map
    # -----------------------------
    # -----------------------------
    # Inset map
    # High-impact journal style: low-saturation land green + light ocean blue
    # -----------------------------
    with fig.inset(
        position="jTL+w3.8c+o0.18c",
        box="+p0.55p,black+gwhite"
    ):
        inset_region = [-124.5, -112.5, 31.8, 41.5]

        fig.coast(
            region=inset_region,
            projection="M3.8c",
            land="#cfdcc8",          # Low-saturation light green (land)
            water="#dbeaf4",         # Light blue (ocean)
            borders="1/0.35p,gray45",
            shorelines="0.35p,gray45",
            frame=False
        )

        # Weaken faults to avoid clutter in the inset map
        fig.plot(
            data=str(FAULT_SHP),
            pen="0.10p,gray40@25"
        )

        # Main-map extent box
        x_box = [REGION[0], REGION[1], REGION[1], REGION[0], REGION[0]]
        y_box = [REGION[2], REGION[2], REGION[3], REGION[3], REGION[2]]
        fig.plot(
            x=x_box,
            y=y_box,
            pen="0.90p,#b22222"
        )

        with pygmt.config(
            MAP_FRAME_TYPE="plain",
            MAP_FRAME_PEN="0.55p,black"
        ):
            fig.basemap(
                region=inset_region,
                projection="M3.8c",
                frame=True
            )

    return fig


# =========================================================
# 4. Main
# =========================================================
def main():
    check_file_exists(EQ_CSV, "Earthquake catalog")
    check_file_exists(STATION_FOLDER, "Station folder")
    check_file_exists(DEM_GRID, "DEM grid")
    check_file_exists(FAULT_SHP, "Fault shapefile")

    eq_df = load_earthquake_catalog(
        csv_path=EQ_CSV,
        depth_range=DEPTH_RANGE,
        mag_min=MAG_MIN,
        size_scale=SIZE_SCALE
    )
    print(f"Loaded {len(eq_df)} earthquakes after filtering")

    station_lons, station_lats = load_station_positions(STATION_FOLDER)
    print(f"Found {len(station_lons)} station positions")

    fig = create_map_figure(eq_df, station_lons, station_lats)

    OUTPUT_PNG.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(str(OUTPUT_PNG), dpi=600)
    fig.savefig(str(OUTPUT_PDF))

    print("Figure saved to:")
    print(OUTPUT_PNG)
    print(OUTPUT_PDF)


if __name__ == "__main__":
    main()
'''

In [ ]:
# (b) Plot station map for the m range
'''
import os
from pathlib import Path
import pandas as pd
import pygmt


# =========================================================
# 1. Global configuration
# =========================================================
REGION = [-119.0999, -116.0993, 34.2695, 37.2695]
PROJECTION = "M15c"

EQ_CSV = Path("D:/a/master/Earthquake-US/Fig-Use/Fig-1/ridgecrest_usgs.csv")
STATION_FOLDER = Path("D:/a/master/Earthquake-US/Data/20190706-102/PosData-7")
DEM_GRID = Path("D:/a/master/Earthquake-US/Fig-Use/Fig-1/DEM/SRTM_Ridgecrest_30m.tif")
FAULT_SHP = Path("D:/a/master/Earthquake-US/Fig-Use/Fig-1/faults/Qfaults_US_Database.shp")

OUTPUT_PNG = Path("D:/a/master/Earthquake-US/Fig-Over-Output/Fig-S6/Map/102_with_stations_gnss_major_final_v2.png")
OUTPUT_PDF = Path("D:/a/master/Earthquake-US/Fig-Over-Output/Fig-S6/Map/102_with_stations_gnss_major_final_v2.pdf")

DEPTH_RANGE = [0, 10]
MAG_MIN = 1.5
SIZE_SCALE = 0.020

PLOT_BACKGROUND_EQ = True

# -----------------------------
# Colors
# -----------------------------
MAIN_64_COLOR = "#900000"
MAIN_71_COLOR = "black"
GNSS_COLOR = "#432818"

# Mainshocks
MAINSHOCKS = [
    {
        "lon": -117.504,
        "lat": 35.705,
        "depth": 8.0,
        "style": "a0.6c",
        "fill": MAIN_64_COLOR,
        "pen": None,
        "label": "Mw 6.4 Depth 8.0 km",
    },
    {
        "lon": -117.599,
        "lat": 35.769,
        "depth": 10.5,
        "style": "a0.8c",
        "fill": MAIN_71_COLOR,
        "pen": None, 
        "label": "Mw 7.1 Depth 10.5 km",
    },
]


# =========================================================
# 2. Utilities
# =========================================================
def check_file_exists(path_obj, description="file"):
    if not path_obj.exists():
        raise FileNotFoundError(f"{description} not found: {path_obj}")


def load_earthquake_catalog(csv_path, depth_range=(0, 10), mag_min=1.5, size_scale=0.015):
    check_file_exists(csv_path, "Earthquake catalog")

    df = pd.read_csv(csv_path)

    required_cols = ["longitude", "latitude", "depth", "mag"]
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in earthquake catalog: {missing}")

    df = df[required_cols].dropna()
    df = df[
        (df["depth"] >= depth_range[0]) &
        (df["depth"] <= depth_range[1]) &
        (df["mag"] >= mag_min)
    ].copy()

    df["size"] = size_scale * df["mag"]
    return df


def load_station_positions(folder_path):
    check_file_exists(folder_path, "Station folder")

    station_lons = []
    station_lats = []

    csv_files = sorted(folder_path.glob("*.csv"))
    if not csv_files:
        print(f"No station CSV files found in: {folder_path}")
        return station_lons, station_lats

    for file_path in csv_files:
        try:
            df = pd.read_csv(file_path)

            if df.empty:
                continue

            required_cols = ["NLat", "Elong"]
            if not all(col in df.columns for col in required_cols):
                print(f"Skipped {file_path.name}: missing columns {required_cols}")
                continue

            station_lats.append(df.iloc[0]["NLat"])
            station_lons.append(df.iloc[0]["Elong"])

        except Exception as e:
            print(f"Error reading {file_path.name}: {e}")

    return station_lons, station_lats


# =========================================================
# 3. Plotting
# =========================================================
def create_map_figure(eq_df, station_lons, station_lats):
    fig = pygmt.Figure()

    # -----------------------------
    # Global style
    # -----------------------------
    pygmt.config(
        FONT="15p,Helvetica,black",
        FONT_ANNOT_PRIMARY="20p,Helvetica,black",
        FONT_LABEL="15p,Helvetica,black",
        FONT_TITLE="15p,Helvetica,black",
        MAP_FRAME_TYPE="fancy",
        MAP_FRAME_PEN="0.7p,black",
        MAP_TICK_PEN_PRIMARY="0.6p,black",
        MAP_TICK_LENGTH_PRIMARY="0.14c",
        MAP_LABEL_OFFSET="0.10c",
        MAP_ANNOT_OFFSET_PRIMARY="0.07c",
        FORMAT_GEO_MAP="dddF",
    )

    # -----------------------------
    # Background topography (paper-style shaded relief)
    # -----------------------------
    pygmt.makecpt(cmap="grayC", series=[-500, 2500], reverse=False)
    
    # Base relief
    fig.grdimage(
        grid=str(DEM_GRID),
        region=REGION,
        projection=PROJECTION,
        cmap=True,
        shading="+a315+nt0.8",
        transparency=8,
    )
    fig.coast(
    region=REGION,
    projection=PROJECTION,
    land="245/245/245@25",
    water="245/245/245@25",
    frame=False
    )

    # -----------------------------
    # Faults (weakened) 
    # -----------------------------
    fig.plot(
        data=str(FAULT_SHP),
        pen="0.12p,gray35@12"
    )

    # -----------------------------
    # CPT only for background earthquakes
    # -----------------------------
    pygmt.makecpt(cmap="hot", series=DEPTH_RANGE, reverse=True, background=True)

    # -----------------------------
    # Background earthquakes 
    # Slightly enlarged, with size still representing magnitude
    # -----------------------------
    if PLOT_BACKGROUND_EQ and len(eq_df) > 0:
        fig.plot(
            x=eq_df["longitude"],
            y=eq_df["latitude"],
            style="c",
            size=eq_df["size"] * 0.90,
            fill=eq_df["depth"],
            cmap=True,
            pen=None,
            transparency=22,
        )

    # -----------------------------
    # GNSS stations
    # -----------------------------
    if station_lons and station_lats:
        fig.plot(
            x=station_lons,
            y=station_lats,
            style="t0.34c",
            pen=f"1.05p,{GNSS_COLOR}",
        )
        print(f"Plotted {len(station_lons)} station positions")
    else:
        print("No station positions found to plot")

    # -----------------------------
    # Mainshocks
    # Fixed colors, not included in the colorbar
    # -----------------------------
    for shock in MAINSHOCKS:
        fig.plot(
            x=[shock["lon"]],
            y=[shock["lat"]],
            style=shock["style"],
            fill=shock["fill"],
            pen=shock["pen"],
        )

    # -----------------------------
    # Basemap
    # -----------------------------
    fig.basemap(
        region=REGION,
        projection=PROJECTION,
        frame=["WSen", "xa1f0.5", "ya1f0.5"]
    )

    # -----------------------------
    # Scale bar
    # -----------------------------
    with pygmt.config(
        FONT_ANNOT_PRIMARY="11p,Helvetica,black",
        FONT_LABEL="12p,Helvetica,black"
    ):
        fig.basemap(
            map_scale="jTR+w20k+o2.25c/3.65c+f+lkm"
        )

    # -----------------------------
    # Inset map
    # -----------------------------
    # -----------------------------
    # Inset map
    # High-impact journal style: low-saturation land green + light ocean blue
    # -----------------------------
    with fig.inset(
        position="jTL+w3.8c+o0.18c",
        box="+p0.55p,black+gwhite"
    ):
        inset_region = [-124.5, -112.5, 31.8, 41.5]

        fig.coast(
            region=inset_region,
            projection="M3.8c",
            land="#cfdcc8",          # Low-saturation light green (land)
            water="#dbeaf4",         # Light blue (ocean)
            borders="1/0.35p,gray45",
            shorelines="0.35p,gray45",
            frame=False
        )

        # Weaken faults to avoid clutter in the inset map
        fig.plot(
            data=str(FAULT_SHP),
            pen="0.10p,gray40@25"
        )

        # Main-map extent box
        x_box = [REGION[0], REGION[1], REGION[1], REGION[0], REGION[0]]
        y_box = [REGION[2], REGION[2], REGION[3], REGION[3], REGION[2]]
        fig.plot(
            x=x_box,
            y=y_box,
            pen="0.90p,#b22222"
        )

        with pygmt.config(
            MAP_FRAME_TYPE="plain",
            MAP_FRAME_PEN="0.55p,black"
        ):
            fig.basemap(
                region=inset_region,
                projection="M3.8c",
                frame=True
            )

    return fig


# =========================================================
# 4. Main
# =========================================================
def main():
    check_file_exists(EQ_CSV, "Earthquake catalog")
    check_file_exists(STATION_FOLDER, "Station folder")
    check_file_exists(DEM_GRID, "DEM grid")
    check_file_exists(FAULT_SHP, "Fault shapefile")

    eq_df = load_earthquake_catalog(
        csv_path=EQ_CSV,
        depth_range=DEPTH_RANGE,
        mag_min=MAG_MIN,
        size_scale=SIZE_SCALE
    )
    print(f"Loaded {len(eq_df)} earthquakes after filtering")

    station_lons, station_lats = load_station_positions(STATION_FOLDER)
    print(f"Found {len(station_lons)} station positions")

    fig = create_map_figure(eq_df, station_lons, station_lats)

    OUTPUT_PNG.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(str(OUTPUT_PNG), dpi=600)
    fig.savefig(str(OUTPUT_PDF))

    print("Figure saved to:")
    print(OUTPUT_PNG)
    print(OUTPUT_PDF)


if __name__ == "__main__":
    main()

'''

In [ ]:
# (c) Plot station map for the l range
'''
import os
from pathlib import Path
import pandas as pd
import pygmt


# =========================================================
# 1. Global configuration
# =========================================================
REGION = [-120.4, -114.4, 32.2, 38.2]
PROJECTION = "M15c"

EQ_CSV = Path("D://a//master//Earthquake-US//Fig-Use//Fig-S6//ridgecrest_usgs.csv")
STATION_FOLDER = Path("D:/a/master/Earthquake-US/Data/20190706-364/PosData-7")
DEM_GRID = Path("D://a//master//Earthquake-US//Fig-Use//Fig-S6//processing//SRTM_Ridgecrest_30m.tif")
FAULT_SHP = Path("D:/a/master/Earthquake-US/Fig-Use/Fig-1/faults/Qfaults_US_Database.shp")

OUTPUT_PNG = Path("D:/a/master/Earthquake-US/Fig-Over-Output/Fig-S6/Map/364_with_stations_gnss.png")
OUTPUT_PDF = Path("D:/a/master/Earthquake-US/Fig-Over-Output/Fig-S6/Map/364_with_stations_gnss.pdf")

DEPTH_RANGE = [0, 10]
MAG_MIN = 1.5
SIZE_SCALE = 0.020

PLOT_BACKGROUND_EQ = True

# -----------------------------
# Colors
# -----------------------------
MAIN_64_COLOR = "#900000"
MAIN_71_COLOR = "black"
GNSS_COLOR = "#432818"

# Mainshocks
MAINSHOCKS = [
    {
        "lon": -117.504,
        "lat": 35.705,
        "depth": 8.0,
        "style": "a0.6c",
        "fill": MAIN_64_COLOR,
        "pen": None,
        "label": "Mw 6.4 Depth 8.0 km",
    },
    {
        "lon": -117.599,
        "lat": 35.769,
        "depth": 10.5,
        "style": "a0.8c",
        "fill": MAIN_71_COLOR,
        "pen": None,
        "label": "Mw 7.1 Depth 10.5 km",
    },
]


# =========================================================
# 2. Utilities
# =========================================================
def check_file_exists(path_obj, description="file"):
    if not path_obj.exists():
        raise FileNotFoundError(f"{description} not found: {path_obj}")


def load_earthquake_catalog(csv_path, depth_range=(0, 10), mag_min=1.5, size_scale=0.015):
    check_file_exists(csv_path, "Earthquake catalog")

    df = pd.read_csv(csv_path)

    required_cols = ["longitude", "latitude", "depth", "mag"]
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in earthquake catalog: {missing}")

    df = df[required_cols].dropna()

    df = df[
        (df["depth"] >= depth_range[0]) &
        (df["depth"] <= depth_range[1]) &
        (df["mag"] >= mag_min)
    ].copy()

    df["size"] = size_scale * df["mag"]

    return df


def load_station_positions(folder_path):
    check_file_exists(folder_path, "Station folder")

    station_lons = []
    station_lats = []

    csv_files = sorted(folder_path.glob("*.csv"))
    if not csv_files:
        print(f"No station CSV files found in: {folder_path}")
        return station_lons, station_lats

    for file_path in csv_files:
        try:
            df = pd.read_csv(file_path)

            if df.empty:
                continue

            required_cols = ["NLat", "Elong"]
            if not all(col in df.columns for col in required_cols):
                print(f"Skipped {file_path.name}: missing columns {required_cols}")
                continue

            station_lats.append(df.iloc[0]["NLat"])
            station_lons.append(df.iloc[0]["Elong"])

        except Exception as e:
            print(f"Error reading {file_path.name}: {e}")

    return station_lons, station_lats


# =========================================================
# 3. Plotting
# =========================================================
def create_map_figure(eq_df, station_lons, station_lats):
    fig = pygmt.Figure()

    # -----------------------------
    # Global style
    # -----------------------------
    pygmt.config(
        FONT="15p,Helvetica,black",
        FONT_ANNOT_PRIMARY="20p,Helvetica,black",
        FONT_LABEL="15p,Helvetica,black",
        FONT_TITLE="15p,Helvetica,black",
        MAP_FRAME_TYPE="fancy",
        MAP_FRAME_PEN="0.7p,black",
        MAP_TICK_PEN_PRIMARY="0.6p,black",
        MAP_TICK_LENGTH_PRIMARY="0.14c",
        MAP_LABEL_OFFSET="0.10c",
        MAP_ANNOT_OFFSET_PRIMARY="0.07c",
        FORMAT_GEO_MAP="dddF",
    )

    # -----------------------------
    # Background topography
    # -----------------------------
    pygmt.makecpt(cmap="grayC", series=[-500, 2500], reverse=False)

    fig.grdimage(
        grid=str(DEM_GRID),
        region=REGION,
        projection=PROJECTION,
        cmap=True,
        shading="+a315+nt0.8",
        transparency=8,
    )

    fig.coast(
        region=REGION,
        projection=PROJECTION,
        land="245/245/245@25",
        water="245/245/245@25",
        frame=False,
    )

    # -----------------------------
    # Faults
    # -----------------------------
    fig.plot(
        data=str(FAULT_SHP),
        pen="0.12p,gray35@12",
    )

    # -----------------------------
    # CPT only for background earthquakes
    # -----------------------------
    pygmt.makecpt(cmap="hot", series=DEPTH_RANGE, reverse=True, background=True)

    # -----------------------------
    # Background earthquakes
    # -----------------------------
    if PLOT_BACKGROUND_EQ and len(eq_df) > 0:
        fig.plot(
            x=eq_df["longitude"],
            y=eq_df["latitude"],
            style="c",
            size=eq_df["size"] * 0.90,
            fill=eq_df["depth"],
            cmap=True,
            pen=None,
            transparency=22,
        )

    # -----------------------------
    # GNSS stations
    # -----------------------------
    if station_lons and station_lats:
        fig.plot(
            x=station_lons,
            y=station_lats,
            style="t0.34c",
            pen=f"1.05p,{GNSS_COLOR}",
        )
        print(f"Plotted {len(station_lons)} station positions")
    else:
        print("No station positions found to plot")

    # -----------------------------
    # Mainshocks
    # -----------------------------
    for shock in MAINSHOCKS:
        fig.plot(
            x=[shock["lon"]],
            y=[shock["lat"]],
            style=shock["style"],
            fill=shock["fill"],
            pen=shock["pen"],
        )

    # -----------------------------
    # Basemap
    # Display only:
    # Latitude: 34°N, 36°N, 38°N
    # Longitude: 119°W, 117°W, 115°W
    # -----------------------------
    fig.basemap(
        region=REGION,
        projection=PROJECTION,
        frame=[
            "WSen",
            "xa2-1",
            "ya2",
        ],
    )

    # -----------------------------
    # Scale bar
    # Keep the native GMT scale bar; do not draw it manually
    # -----------------------------
    with pygmt.config(
        FONT_ANNOT_PRIMARY="11p,Helvetica,black",
        FONT_LABEL="12p,Helvetica,black",
    ):
        fig.basemap(
           map_scale="jTR+w40k+o2.25c/3.65c+f+lkm",
        )

    # -----------------------------
    # Inset map
    # -----------------------------
    with fig.inset(
        position="jTL+w3.8c+o0.18c",
        box="+p0.55p,black+gwhite",
    ):
        inset_region = [-124.5, -112.5, 31.8, 41.5]

        fig.coast(
            region=inset_region,
            projection="M3.8c",
            land="#cfdcc8",
            water="#dbeaf4",
            borders="1/0.35p,gray45",
            shorelines="0.35p,gray45",
            frame=False,
        )

        # Weaken faults to avoid clutter in the inset map
        fig.plot(
            data=str(FAULT_SHP),
            pen="0.10p,gray40@25",
        )

        # Main-map extent box
        x_box = [REGION[0], REGION[1], REGION[1], REGION[0], REGION[0]]
        y_box = [REGION[2], REGION[2], REGION[3], REGION[3], REGION[2]]

        fig.plot(
            x=x_box,
            y=y_box,
            pen="0.90p,#b22222",
        )

        with pygmt.config(
            MAP_FRAME_TYPE="plain",
            MAP_FRAME_PEN="0.55p,black",
        ):
            fig.basemap(
                region=inset_region,
                projection="M3.8c",
                frame=True,
            )

    return fig


# =========================================================
# 4. Main
# =========================================================
def main():
    check_file_exists(EQ_CSV, "Earthquake catalog")
    check_file_exists(STATION_FOLDER, "Station folder")
    check_file_exists(DEM_GRID, "DEM grid")
    check_file_exists(FAULT_SHP, "Fault shapefile")

    eq_df = load_earthquake_catalog(
        csv_path=EQ_CSV,
        depth_range=DEPTH_RANGE,
        mag_min=MAG_MIN,
        size_scale=SIZE_SCALE,
    )

    print(f"Loaded {len(eq_df)} earthquakes after filtering")

    station_lons, station_lats = load_station_positions(STATION_FOLDER)

    print(f"Found {len(station_lons)} station positions")

    fig = create_map_figure(eq_df, station_lons, station_lats)

    OUTPUT_PNG.parent.mkdir(parents=True, exist_ok=True)

    fig.savefig(str(OUTPUT_PNG), dpi=600)
    fig.savefig(str(OUTPUT_PDF))

    print("Figure saved to:")
    print(OUTPUT_PNG)
    print(OUTPUT_PDF)


if __name__ == "__main__":
    main()
'''

In [ ]:
# (d)-(f) First eigenvalue, entropy, and entropy derivative for the three regions
'''
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import AutoMinorLocator
from pathlib import Path

# =========================
# 1. Input and output paths
# =========================
input_file = Path(r'D:\a\master\Earthquake-US\Fig-Over-Output\Fig-S6\W&S&Phi\IMS_eigenvalues_entropy_data.xlsx')
output_dir = Path(r'D:\a\master\Earthquake-US\Fig-Over-Output\Fig-S6\W&S&Phi')
output_dir.mkdir(parents=True, exist_ok=True)

out_png = output_dir / 'W1_S_deltaS_nature_two_column.png'
out_pdf = output_dir / 'W1_S_deltaS_nature_two_column.pdf'

# Time range used for plotting
PLOT_START_DATE = pd.to_datetime("2018-09-06")
PLOT_END_DATE = pd.to_datetime("2020-05-06")

# Display only two major ticks on the x-axis
XTICK_DATES = [
    pd.to_datetime("2019-02-01"),
    pd.to_datetime("2019-12-01")
]
XTICK_LABELS = ["2019-02", "2019-12"]

# =========================
# 2. Global parameters for Nature style
# =========================
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['mathtext.fontset'] = 'stix'
plt.rcParams['font.size'] = 7
plt.rcParams['axes.labelsize'] = 8
plt.rcParams['axes.titlesize'] = 8
plt.rcParams['xtick.labelsize'] = 7
plt.rcParams['ytick.labelsize'] = 7
plt.rcParams['legend.fontsize'] = 7
plt.rcParams['axes.linewidth'] = 0.7
plt.rcParams['xtick.major.width'] = 0.7
plt.rcParams['ytick.major.width'] = 0.7
plt.rcParams['xtick.minor.width'] = 0.5
plt.rcParams['ytick.minor.width'] = 0.5
plt.rcParams['xtick.major.size'] = 3.5
plt.rcParams['ytick.major.size'] = 3.5
plt.rcParams['xtick.minor.size'] = 2.0
plt.rcParams['ytick.minor.size'] = 2.0
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

# Set the background to white
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'

# =========================
# 3. Overall figure size
# Nature double-column width is approximately 183 mm = 7.2 inch
# Three subplots arranged horizontally
# =========================
FIG_WIDTH = 7.2
FIG_HEIGHT = 2.35

# =========================
# 4. Colors
# =========================
colors = ['#1f77b4', '#d95f02', '#1b9e77']

# =========================
# 5. Load data
# =========================
def load_data(input_file):
    df = pd.read_excel(input_file)
    df.columns = df.columns.str.strip()
    df['date'] = pd.to_datetime(df['date'])

    required_cols = [
        'date',
        'lambda_1_20', 'lambda_1_102', 'lambda_1_364',
        'entropy_20', 'entropy_102', 'entropy_364',
        'entropy_change_rate_20', 'entropy_change_rate_102', 'entropy_change_rate_364'
    ]

    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f'The following columns do not exist in the file: {missing_cols}')

    # Use only data from 2018-09-06 to 2020-05-06
    df = df[
        (df['date'] >= PLOT_START_DATE) &
        (df['date'] <= PLOT_END_DATE)
    ].copy()

    df = df.sort_values('date').reset_index(drop=True)

    if df.empty:
        raise ValueError('The filtered data are empty. Please check the date range or the original data.')

    return df

# =========================
# 6. Function for plotting a single panel
# =========================
def plot_panel(
    ax,
    df,
    ycols,
    labels,
    ylabel,
    main_quake_date,
    legend_loc='lower right',
    show_legend=True,
    show_main_quake_in_legend=False,
    yticks=None,
    yticklabels=None
):
    # White background
    ax.set_facecolor('white')
    ax.patch.set_alpha(1.0)

    # Three curves
    for ycol, color, label in zip(ycols, colors, labels):
        ax.plot(
            df['date'],
            df[ycol],
            color=color,
            linewidth=1.2,
            label=label,
            solid_capstyle='round'
        )

    # Vertical dashed line for the mainshock
    if show_main_quake_in_legend:
        ax.axvline(
            main_quake_date,
            color='black',
            linestyle='--',
            linewidth=1.0,
            label='Main',
            zorder=3
        )
    else:
        ax.axvline(
            main_quake_date,
            color='black',
            linestyle='--',
            linewidth=1.0,
            zorder=3
        )

    # Axis labels
    ax.set_ylabel(ylabel)
    ax.set_xlabel('Date')

    # Fix the x-axis range from 2018-09-06 to 2020-05-06
    ax.set_xlim(PLOT_START_DATE, PLOT_END_DATE)

    # Display only two major tick labels on the x-axis: 2019-02 and 2019-12
    ax.set_xticks(XTICK_DATES)
    ax.set_xticklabels(XTICK_LABELS)

    # Retain x-axis minor ticks to show scale details; no labels are displayed
    ax.xaxis.set_minor_locator(mdates.MonthLocator(interval=1))

    # Display only the three specified y-axis values
    if yticks is not None:
        ax.set_yticks(yticks)

    if yticklabels is not None:
        ax.set_yticklabels(yticklabels)

    # Y-axis minor ticks
    ax.yaxis.set_minor_locator(AutoMinorLocator(2))

    # Major ticks: displayed on all four directions
    ax.tick_params(
        axis='both',
        which='major',
        direction='in',
        length=3.5,
        width=0.7,
        top=False,
        right=False,
        bottom=True,
        left=True,
        color='black',
        labelcolor='black'
    )

    # Minor ticks: displayed on all four directions
    ax.tick_params(
        axis='both',
        which='minor',
        direction='in',
        length=2.0,
        width=0.5,
        top=False,
        right=False,
        bottom=True,
        left=True,
        color='black'
    )

    # Keep tick labels only on the bottom and left sides
    ax.tick_params(labeltop=False, labelright=False)

    # Do not rotate date labels
    plt.setp(ax.get_xticklabels(), rotation=0, ha='center')

    # Force all four borders to be displayed
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.7)
        spine.set_color('black')

    # No background grid
    ax.grid(False)

    # Reduce horizontal margins
    ax.margins(x=0.005)

    # Legend
    if show_legend:
        ax.legend(
            loc=legend_loc,
            frameon=False,
            handlelength=2.0,
            borderpad=0.2,
            labelspacing=0.3
        )

# =========================
# 7. Main function
# =========================
def main():
    df = load_data(input_file)
    main_quake_date = pd.to_datetime("2019-07-06")

    fig, axes = plt.subplots(1, 3, figsize=(FIG_WIDTH, FIG_HEIGHT), sharex=False)

    # White background for the entire figure
    fig.patch.set_alpha(1.0)
    fig.patch.set_facecolor('white')

    # -------- Panel 1: W1 --------
    plot_panel(
        ax=axes[0],
        df=df,
        ycols=['lambda_1_20', 'lambda_1_102', 'lambda_1_364'],
        labels=[r'$W^{1}_{s}$', r'$W^{1}_{m}$', r'$W^{1}_{l}$'],
        ylabel=r'$W^{1}$',
        main_quake_date=main_quake_date,
        legend_loc='upper right',
        show_legend=True,
        show_main_quake_in_legend=True,
        yticks=[0.2, 0.5, 0.8],
        yticklabels=['0.2', '0.5', '0.8']
    )

    # -------- Panel 2: S --------
    plot_panel(
        ax=axes[1],
        df=df,
        ycols=['entropy_20', 'entropy_102', 'entropy_364'],
        labels=[r'$S_{s}$', r'$S_{m}$', r'$S_{l}$'],
        ylabel=r'$S$',
        main_quake_date=main_quake_date,
        legend_loc='lower right',
        show_legend=True,
        show_main_quake_in_legend=False,
        yticks=[1.0, 2.0, 3.0],
        yticklabels=['1.0', '2.0', '3.0']
    )

    # -------- Panel 3: ΔS --------
    plot_panel(
        ax=axes[2],
        df=df,
        ycols=['entropy_change_rate_20', 'entropy_change_rate_102', 'entropy_change_rate_364'],
        labels=[r'$\Delta S_{s}$', r'$\Delta S_{m}$', r'$\Delta S_{l}$'],
        ylabel=r'$\Delta S$',
        main_quake_date=main_quake_date,
        legend_loc='lower right',
        show_legend=True,
        show_main_quake_in_legend=False,
        yticks=[-0.6, 0.0, 0.6],
        yticklabels=['-0.6', '0.0', '0.6']
    )

    # Fine-tune layout
    plt.subplots_adjust(
        left=0.06,
        right=0.995,
        bottom=0.24,
        top=0.96,
        wspace=0.28
    )

    # Save with white background
    fig.savefig(out_png, dpi=600, bbox_inches='tight', facecolor='white')
    fig.savefig(out_pdf, bbox_inches='tight', facecolor='white')
    plt.close(fig)

    print('Plotting completed. Files saved to:')
    print(out_png)
    print(out_pdf)

# =========================
# 8. Run
# =========================
if __name__ == '__main__':
    main()
'''

Fig-S7

In [ ]:
# (a)-(c) Three Phi values for the three regions
'''
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import AutoMinorLocator
from pathlib import Path

# =========================
# 1. Input and output paths
# =========================
input_file = Path(r'D:\a\master\Earthquake-US\Fig-Over-Output\Fig-S7\IMS_phi_values_data.xlsx')
output_dir = Path(r'D:\a\master\Earthquake-US\Fig-Over-Output\Fig-S7')
output_dir.mkdir(parents=True, exist_ok=True)

out_png = output_dir / 'Phi_nature_two_column.png'
out_pdf = output_dir / 'Phi_nature_two_column.pdf'

# Time range used for plotting
PLOT_START_DATE = pd.to_datetime("2018-09-06")
PLOT_END_DATE = pd.to_datetime("2020-05-06")

# Major ticks displayed on the x-axis
XTICK_DATES = [
    pd.to_datetime("2018-11-01"),
    pd.to_datetime("2019-03-01"),
    pd.to_datetime("2019-07-01"),
    pd.to_datetime("2019-11-01"),
    pd.to_datetime("2020-03-01"),
]

XTICK_LABELS = [
    "2018-11",
    "2019-03",
    "2019-07",
    "2019-11",
    "2020-03",
]

# =========================
# 2. Global parameters for Nature style
# =========================
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['mathtext.fontset'] = 'stix'
plt.rcParams['font.size'] = 10
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['xtick.labelsize'] = 9
plt.rcParams['ytick.labelsize'] = 9
plt.rcParams['legend.fontsize'] = 9
plt.rcParams['axes.linewidth'] = 0.7
plt.rcParams['xtick.major.width'] = 0.7
plt.rcParams['ytick.major.width'] = 0.7
plt.rcParams['xtick.minor.width'] = 0.5
plt.rcParams['ytick.minor.width'] = 0.5
plt.rcParams['xtick.major.size'] = 3.5
plt.rcParams['ytick.major.size'] = 3.5
plt.rcParams['xtick.minor.size'] = 2.0
plt.rcParams['ytick.minor.size'] = 2.0
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

# Use a white background
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'

# =========================
# 3. Overall figure size
# =========================
FIG_WIDTH = 7.2
FIG_HEIGHT = 6

# =========================
# 4. Colors
# =========================
colors = ['#1f77b4', '#d95f02', '#1b9e77']

# =========================
# 5. Load data
# =========================
def load_phi_data(input_file):
    df = pd.read_excel(input_file)
    df.columns = df.columns.str.strip()
    df['date'] = pd.to_datetime(df['date'])

    required_cols = [
        'date',
        'global_phi_20', 'global_phi_102', 'global_phi_364',
        'average_phi_20', 'average_phi_102', 'average_phi_364',
        'diff_phi_20', 'diff_phi_102', 'diff_phi_364'
    ]

    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f'The following columns do not exist in the file: {missing_cols}')

    # Use only data from 2018-09-06 to 2020-05-06
    df = df[
        (df['date'] >= PLOT_START_DATE) &
        (df['date'] <= PLOT_END_DATE)
    ].copy()

    df = df.sort_values('date').reset_index(drop=True)

    if df.empty:
        raise ValueError('The filtered data are empty. Please check the date range or the original data.')

    return df

# =========================
# 6. Function for plotting a single panel
# =========================
def plot_panel(
    ax,
    df,
    ycols,
    labels,
    ylabel,
    main_quake_date,
    legend_loc='lower right',
    show_legend=True,
    show_main_quake_in_legend=False,
    show_xlabel=True,
    legend_bbox_to_anchor=None,
    yticks=None,
    yticklabels=None
):
    # White background
    ax.set_facecolor('white')
    ax.patch.set_alpha(1.0)

    # Three curves
    for ycol, color, label in zip(ycols, colors, labels):
        ax.plot(
            df['date'],
            df[ycol],
            color=color,
            linewidth=1.2,
            label=label,
            solid_capstyle='round'
        )

    # Vertical dashed line for the mainshock
    if show_main_quake_in_legend:
        ax.axvline(
            main_quake_date,
            color='black',
            linestyle='--',
            linewidth=1.0,
            label='Main',
            zorder=3
        )
    else:
        ax.axvline(
            main_quake_date,
            color='black',
            linestyle='--',
            linewidth=1.0,
            zorder=3
        )

    # Axis labels
    ax.set_ylabel(ylabel)

    if show_xlabel:
        ax.set_xlabel('Date')
        ax.tick_params(axis='x', labelbottom=True)
    else:
        ax.set_xlabel('')
        ax.tick_params(axis='x', labelbottom=False)

    # Fix the x-axis range
    ax.set_xlim(PLOT_START_DATE, PLOT_END_DATE)

    # Display only the specified 5 major ticks on the x-axis
    ax.set_xticks(XTICK_DATES)
    ax.set_xticklabels(XTICK_LABELS)

    # X-axis minor ticks: one minor tick every month
    ax.xaxis.set_minor_locator(mdates.MonthLocator(interval=1))

    # Display only the three specified y-axis values
    if yticks is not None:
        ax.set_yticks(yticks)

    if yticklabels is not None:
        ax.set_yticklabels(yticklabels)

    # Y-axis minor ticks
    ax.yaxis.set_minor_locator(AutoMinorLocator(3))

    # Keep borders on all four sides
    for side in ['left', 'right', 'top', 'bottom']:
        ax.spines[side].set_visible(True)
        ax.spines[side].set_linewidth(0.7)
        ax.spines[side].set_color('black')

    # Display major ticks only on the left and bottom
    ax.tick_params(
        axis='both',
        which='major',
        direction='in',
        length=3.5,
        width=0.7,
        top=False,
        bottom=True,
        left=True,
        right=False,
        labeltop=False,
        labelright=False,
        color='black',
        labelcolor='black'
    )

    # Display minor ticks only on the left and bottom
    ax.tick_params(
        axis='both',
        which='minor',
        direction='in',
        length=2.0,
        width=0.5,
        top=False,
        bottom=True,
        left=True,
        right=False,
        color='black'
    )

    # Do not rotate date labels
    plt.setp(ax.get_xticklabels(), rotation=0, ha='center')

    # No grid
    ax.grid(False)

    # Reduce horizontal margins
    ax.margins(x=0.005)

    # Legend
    if show_legend:
        ax.legend(
            loc=legend_loc,
            bbox_to_anchor=legend_bbox_to_anchor,
            frameon=False,
            handlelength=2.0,
            borderpad=0.2,
            labelspacing=0.3
        )

# =========================
# 7. Main function: plot only the three Phi panels
# =========================
def main():
    df = load_phi_data(input_file)
    main_quake_date = pd.to_datetime("2019-07-06")

    fig, axes = plt.subplots(3, 1, figsize=(FIG_WIDTH, FIG_HEIGHT), sharex=True)

    # White background for the entire figure
    fig.patch.set_facecolor('white')
    fig.patch.set_alpha(1.0)

    # -------- Panel 1: global phi --------
    plot_panel(
        ax=axes[0],
        df=df,
        ycols=['global_phi_20', 'global_phi_102', 'global_phi_364'],
        labels=[
            r'$\Phi_{\mathrm{global},s}$',
            r'$\Phi_{\mathrm{global},m}$',
            r'$\Phi_{\mathrm{global},l}$'
        ],
        ylabel=r'$\Phi_{\mathrm{global}}$',
        main_quake_date=main_quake_date,
        legend_loc='lower left',
        show_legend=True,
        show_main_quake_in_legend=True,
        show_xlabel=False,
        legend_bbox_to_anchor=(0.09, -0.02),
        yticks=[0.2, 0.5, 0.8],
        yticklabels=['0.2', '0.5', '0.8']
    )

    # -------- Panel 2: average phi --------
    plot_panel(
        ax=axes[1],
        df=df,
        ycols=['average_phi_20', 'average_phi_102', 'average_phi_364'],
        labels=[
            r'$\overline{\Phi}_{\mathrm{regional},s}$',
            r'$\overline{\Phi}_{\mathrm{regional},m}$',
            r'$\overline{\Phi}_{\mathrm{regional},l}$'
        ],
        ylabel=r'$\overline{\Phi}_{\mathrm{regional}}$',
        main_quake_date=main_quake_date,
        legend_loc='lower left',
        show_legend=True,
        show_main_quake_in_legend=False,
        show_xlabel=False,
        yticks=[0.3, 0.6, 0.9],
        yticklabels=['0.3', '0.6', '0.9']
    )

    # -------- Panel 3: diff phi --------
    plot_panel(
        ax=axes[2],
        df=df,
        ycols=['diff_phi_20', 'diff_phi_102', 'diff_phi_364'],
        labels=[
            r'$\Delta \Phi_{rg,s}$',
            r'$\Delta \Phi_{rg,m}$',
            r'$\Delta \Phi_{rg,l}$'
        ],
        ylabel=r'$\Delta \Phi_{rg}$',
        main_quake_date=main_quake_date,
        legend_loc='upper right',
        show_legend=True,
        show_main_quake_in_legend=False,
        show_xlabel=True,
        yticks=[0.2, 0.5, 0.8],
        yticklabels=['0.2', '0.5', '0.8']
    )

    # Uniformly confirm the x-axis range and ticks
    for ax in axes:
        ax.set_xlim(PLOT_START_DATE, PLOT_END_DATE)
        ax.set_xticks(XTICK_DATES)
        ax.set_xticklabels(XTICK_LABELS)

    # Display x-axis labels only on the bottom subplot
    axes[0].tick_params(axis='x', labelbottom=False)
    axes[1].tick_params(axis='x', labelbottom=False)
    axes[2].tick_params(axis='x', labelbottom=True)

    # Reduce the spacing between the three panels
    plt.subplots_adjust(
        left=0.16,
        right=0.98,
        bottom=0.08,
        top=0.98,
        hspace=0.03
    )

    # Save with white background
    fig.savefig(out_png, dpi=600, bbox_inches='tight', facecolor='white')
    fig.savefig(out_pdf, bbox_inches='tight', facecolor='white')
    plt.close(fig)

    print('Plotting completed. Files saved to:')
    print(out_png)
    print(out_pdf)

# =========================
# 8. Run
# =========================
if __name__ == '__main__':
    main()
'''

Fig-S8

In [ ]:
# Data
'''
import math
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import AutoMinorLocator
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.colors import Normalize


# ============================================================
# 1. Modifiable parameters
# ============================================================

input_folder = Path(r"D:\a\master\Earthquake-US\Data\20190706-102\PosData-7")

output_folder = Path(r"D:\a\master\Earthquake-US\Fig-Use-2\Fig-new\Line_Profile_15deg_All_BlueByDistance")
output_folder.mkdir(parents=True, exist_ok=True)

start_date = "2019-07-01"
end_date = "2019-07-08"

center_lat = 35.7695
center_lon = 242.4006667

angle_step = 15
angles = list(range(0, 360, angle_step))

# Number of stations nearest to the ray/line for each direction
stations_per_angle = 7

# True: ray extending outward from the center point
# False: complete line passing through the center point
use_ray = True

# False: plot raw dE/dN
# True: subtract the first-day value of each station
zero_by_first_day = False

save_single_png = True
save_single_pdf = True
save_merged_pdf = True

merged_dE_pdf_path = output_folder / "All_dE_profiles_15deg_blue_by_distance.pdf"
merged_dN_pdf_path = output_folder / "All_dN_profiles_15deg_blue_by_distance.pdf"


# ============================================================
# 2. Plotting style
# ============================================================

def set_plot_style():
    plt.rcParams["font.family"] = "Arial"
    plt.rcParams["font.size"] = 8
    plt.rcParams["axes.labelsize"] = 11
    plt.rcParams["xtick.labelsize"] = 10
    plt.rcParams["ytick.labelsize"] = 10
    plt.rcParams["legend.fontsize"] = 7
    plt.rcParams["pdf.fonttype"] = 42
    plt.rcParams["ps.fonttype"] = 42


# ============================================================
# 3. Coordinate and distance calculations
# ============================================================

def normalize_lon_diff(lon, lon0):
    """
    Calculate the longitude difference, compatible with both
    0~360 and -180~180 longitude formats.
    For example, 242.4°E is equivalent to -117.6°.
    """
    return (lon - lon0 + 180) % 360 - 180


def latlon_to_local_xy_km(lat, lon, lat0, lon0):
    """
    Convert latitude and longitude to local planar coordinates.

    x_km: positive eastward
    y_km: positive northward
    """
    earth_radius_km = 6371.0

    dlat_rad = math.radians(lat - lat0)
    dlon_rad = math.radians(normalize_lon_diff(lon, lon0))

    lat0_rad = math.radians(lat0)

    x_km = earth_radius_km * math.cos(lat0_rad) * dlon_rad
    y_km = earth_radius_km * dlat_rad

    return x_km, y_km


def compute_line_distance(x, y, angle_deg):
    """
    Calculate the distance from a station to the ray/line
    in the specified direction.

    angle_deg:
        Due east is 0°, with counterclockwise positive.
        90° is due north, 180° is due west, and 270° is due south.

    along_km:
        Projected distance along the specified direction.

    perp_km:
        Perpendicular distance to the directional line.
    """
    theta = math.radians(angle_deg)

    ux = math.cos(theta)
    uy = math.sin(theta)

    along_km = x * ux + y * uy
    perp_km = abs(-x * uy + y * ux)

    return along_km, perp_km


# ============================================================
# 4. Read station metadata
# ============================================================

def read_station_metadata(csv_path):
    """
    Read latitude and longitude information for a single station.
    The station name is taken from the filename, e.g., P595.csv -> P595.
    """
    df = pd.read_csv(csv_path)

    required_cols = ["YYYYMMDD", "dE", "dN", "NLat", "Elong"]

    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"{csv_path.name} is missing the required column: {col}")

    station_name = csv_path.stem

    lat = df["NLat"].dropna().iloc[0]
    lon = df["Elong"].dropna().iloc[0]

    x_km, y_km = latlon_to_local_xy_km(
        lat=lat,
        lon=lon,
        lat0=center_lat,
        lon0=center_lon
    )

    # Distance to the center point
    center_distance_km = math.sqrt(x_km ** 2 + y_km ** 2)

    return {
        "station": station_name,
        "csv_path": csv_path,
        "lat": lat,
        "lon": lon,
        "x_km": x_km,
        "y_km": y_km,
        "center_distance_km": center_distance_km
    }


def read_all_station_metadata(input_folder):
    """Read station information from all CSV files in the folder"""
    csv_files = sorted(input_folder.glob("*.csv"))

    if len(csv_files) == 0:
        raise FileNotFoundError(f"No CSV files found in folder: {input_folder}")

    station_infos = []

    for csv_path in csv_files:
        try:
            info = read_station_metadata(csv_path)
            station_infos.append(info)
        except Exception as e:
            print(f"Skipping {csv_path.name}, reason: {e}")

    if len(station_infos) == 0:
        raise RuntimeError("No stations were successfully read. Please check the CSV column names.")

    return pd.DataFrame(station_infos)


# ============================================================
# 5. Select stations for each angle
# ============================================================

def select_stations_for_angle(station_df, angle_deg):
    """
    For a given angle, select the stations_per_angle stations
    nearest to the corresponding ray/line.
    """
    df = station_df.copy()

    along_list = []
    perp_list = []

    for _, row in df.iterrows():
        along_km, perp_km = compute_line_distance(
            x=row["x_km"],
            y=row["y_km"],
            angle_deg=angle_deg
        )

        along_list.append(along_km)
        perp_list.append(perp_km)

    df["angle_deg"] = angle_deg
    df["along_km"] = along_list
    df["perp_km"] = perp_list

    if use_ray:
        df = df[df["along_km"] >= 0].copy()

    df = df.sort_values(["perp_km", "along_km"]).reset_index(drop=True)

    selected_df = df.head(stations_per_angle).copy()

    # Sort by distance along the ray for plotting
    selected_df = selected_df.sort_values("along_km").reset_index(drop=True)

    return selected_df


# ============================================================
# 6. Read time series
# ============================================================

def load_station_timeseries(csv_path):
    """Read dE/dN for a single station within the specified time period"""
    df = pd.read_csv(csv_path)

    df["YYYYMMDD"] = pd.to_datetime(df["YYYYMMDD"], format="%Y%m%d")
    df = df.set_index("YYYYMMDD")
    df = df.loc[start_date:end_date].copy()

    return df


# ============================================================
# 7. Distance color mapping
# ============================================================

def get_blue_color_by_distance(distance_km, norm):
    """
    Return a blue color according to the distance from the station
    to the center point.

    Shorter distance: darker blue
    Longer distance: lighter blue
    """
    cmap = plt.cm.Blues_r

    value = norm(distance_km)

    # Prevent the farthest lines from becoming too light to see
    # In Blues_r, smaller values are darker and larger values are lighter
    value = 0.15 + 0.65 * value

    return cmap(value)


# ============================================================
# 8. Plot dE or dN for a single angle
# ============================================================

def plot_component_for_angle(selected_df, component, angle_deg, distance_norm):
    """
    Plot dE or dN curves for multiple stations at a given angle.

    component:
        "dE" or "dN"
    """
    if selected_df.empty:
        print(f"No available stations at {angle_deg}°; skipping {component}")
        return None

    max_perp_km = selected_df["perp_km"].max()

    fig, ax = plt.subplots(figsize=(6.8, 3.4), dpi=300)

    for _, row in selected_df.iterrows():
        station = row["station"]
        csv_path = row["csv_path"]

        along_km = row["along_km"]
        perp_km = row["perp_km"]
        center_distance_km = row["center_distance_km"]

        df = load_station_timeseries(csv_path)

        if df.empty:
            print(f"Skipping {station}: no data from {start_date} to {end_date}")
            continue

        y = df[component].copy()

        if zero_by_first_day:
            y = y - y.iloc[0]

        color = get_blue_color_by_distance(center_distance_km, distance_norm)

        label = (
            f"{station}  "
            f"R={center_distance_km:.1f} km, "
            f"L={along_km:.1f} km, "
            f"D={perp_km:.1f} km"
        )

        ax.plot(
            df.index,
            y,
            marker="o",
            markersize=4.5,
            linewidth=1.5,
            color=color,
            markerfacecolor=color,
            markeredgecolor=color,
            label=label
        )

    ax.set_xlabel("Date (2019-07)")

    if zero_by_first_day:
        ax.set_ylabel(f"{component} relative displacement (m)")
    else:
        ax.set_ylabel(f"{component} displacement (m)")

    ax.set_title(
        f"{component}, angle = {angle_deg}°, "
        f"nearest {len(selected_df)} stations, "
        f"max line distance = {max_perp_km:.2f} km",
        fontsize=11
    )

    ax.xaxis.set_major_locator(mdates.DayLocator(interval=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
    ax.xaxis.set_minor_locator(mdates.HourLocator(interval=12))

    ax.yaxis.set_minor_locator(AutoMinorLocator(5))

    ax.tick_params(
        axis="x",
        which="major",
        direction="in",
        length=4,
        width=0.8,
        top=True,
        bottom=True
    )

    ax.tick_params(
        axis="x",
        which="minor",
        direction="in",
        length=2.2,
        width=0.7,
        top=True,
        bottom=True
    )

    ax.tick_params(
        axis="y",
        which="major",
        direction="in",
        length=4,
        width=0.8,
        left=True,
        right=True
    )

    ax.tick_params(
        axis="y",
        which="minor",
        direction="in",
        length=2.2,
        width=0.7,
        left=True,
        right=True
    )

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.9)
        spine.set_color("black")

    ax.grid(False)

    ax.legend(
        loc="best",
        frameon=False,
        fontsize=6.5,
        handlelength=2.0
    )

    # Add colorbar: represents distance to the center point
    sm = plt.cm.ScalarMappable(
        cmap=plt.cm.Blues_r,
        norm=distance_norm
    )
    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        ax=ax,
        pad=0.02,
        fraction=0.045
    )
    cbar.set_label("Distance to center (km)", labelpad=5)
    cbar.ax.invert_yaxis()  # Display the darker blue for shorter distances at the top

    cbar.ax.tick_params(
        direction="in",
        length=3,
        width=0.7
    )

    plt.tight_layout()

    return fig


# ============================================================
# 9. Main function: iterate from 0° to 345°
# ============================================================

def main():
    set_plot_style()

    station_df = read_all_station_metadata(input_folder)

    min_dist = station_df["center_distance_km"].min()
    max_dist = station_df["center_distance_km"].max()

    distance_norm = Normalize(vmin=min_dist, vmax=max_dist)

    print("\n================================================")
    print("Distance range from all stations to the center point")
    print("================================================")
    print(f"Minimum distance: {min_dist:.2f} km")
    print(f"Maximum distance: {max_dist:.2f} km")
    print("Color rule: stations closer to the center are darker blue; stations farther from the center are lighter blue.")

    summary_records = []

    pdf_dE = PdfPages(merged_dE_pdf_path) if save_merged_pdf else None
    pdf_dN = PdfPages(merged_dN_pdf_path) if save_merged_pdf else None

    try:
        for angle_deg in angles:
            print("\n================================================")
            print(f"Processing angle: {angle_deg}°")
            print("Angle definition: due east is 0°, increasing counterclockwise")
            print("================================================")

            selected_df = select_stations_for_angle(station_df, angle_deg)

            if selected_df.empty:
                print(f"No available stations in the {angle_deg}° direction; skipping.")
                continue

            max_perp_km = selected_df["perp_km"].max()

            print(f"Number of selected stations: {len(selected_df)}")
            print(f"Maximum perpendicular distance: {max_perp_km:.2f} km")
            print(
                selected_df[
                    [
                        "station",
                        "lat",
                        "lon",
                        "center_distance_km",
                        "along_km",
                        "perp_km"
                    ]
                ].to_string(index=False)
            )

            selected_csv_path = output_folder / f"selected_stations_{angle_deg:03d}deg.csv"
            selected_df.to_csv(selected_csv_path, index=False, encoding="utf-8-sig")

            for _, row in selected_df.iterrows():
                summary_records.append({
                    "angle_deg": angle_deg,
                    "station": row["station"],
                    "lat": row["lat"],
                    "lon": row["lon"],
                    "center_distance_km": row["center_distance_km"],
                    "along_km": row["along_km"],
                    "perp_km": row["perp_km"]
                })

            fig_de = plot_component_for_angle(
                selected_df=selected_df,
                component="dE",
                angle_deg=angle_deg,
                distance_norm=distance_norm
            )

            fig_dn = plot_component_for_angle(
                selected_df=selected_df,
                component="dN",
                angle_deg=angle_deg,
                distance_norm=distance_norm
            )

            if fig_de is not None:
                if save_single_png:
                    fig_de.savefig(
                        output_folder / f"dE_profile_{angle_deg:03d}deg_blue_by_distance.png",
                        dpi=600,
                        bbox_inches="tight"
                    )

                if save_single_pdf:
                    fig_de.savefig(
                        output_folder / f"dE_profile_{angle_deg:03d}deg_blue_by_distance.pdf",
                        bbox_inches="tight"
                    )

                if pdf_dE is not None:
                    pdf_dE.savefig(fig_de, bbox_inches="tight")

                plt.close(fig_de)

            if fig_dn is not None:
                if save_single_png:
                    fig_dn.savefig(
                        output_folder / f"dN_profile_{angle_deg:03d}deg_blue_by_distance.png",
                        dpi=600,
                        bbox_inches="tight"
                    )

                if save_single_pdf:
                    fig_dn.savefig(
                        output_folder / f"dN_profile_{angle_deg:03d}deg_blue_by_distance.pdf",
                        bbox_inches="tight"
                    )

                if pdf_dN is not None:
                    pdf_dN.savefig(fig_dn, bbox_inches="tight")

                plt.close(fig_dn)

    finally:
        if pdf_dE is not None:
            pdf_dE.close()

        if pdf_dN is not None:
            pdf_dN.close()

    summary_df = pd.DataFrame(summary_records)
    summary_csv_path = output_folder / "All_selected_stations_summary_blue_by_distance.csv"
    summary_df.to_csv(summary_csv_path, index=False, encoding="utf-8-sig")

    print("\nAll completed!")
    print(f"Output folder: {output_folder}")
    print(f"Summary table of selected stations for all angles: {summary_csv_path}")

    if save_merged_pdf:
        print(f"Merged dE PDF: {merged_dE_pdf_path}")
        print(f"Merged dN PDF: {merged_dN_pdf_path}")


if __name__ == "__main__":
    main()
'''

In [ ]:
# Curve showing how the epicentral data of each station along the line vary with distance from the epicenter
'''
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import AutoMinorLocator, MaxNLocator
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.cm import ScalarMappable
from pathlib import Path
import numpy as np


# ============================================================
# 1. Input and output paths
# ============================================================

data_folder = Path(
    r"D:\a\master\Earthquake-US\Fig-Use-2\Fig-new\Line_Profile_15deg_All_BlueByDistance"
)

output_png = Path(
    r"D:\a\master\Earthquake-US\Fig-Over-Output\Fig-S8\dE150_dN090_2x1_profile.png"
)
output_pdf = Path(
    r"D:\a\master\Earthquake-US\Fig-Over-Output\Fig-S8\dE150_dN090_2x1_profile.pdf"
)

# Generated station-selection files
selected_dE_csv = data_folder / "selected_stations_150deg.csv"
selected_dN_csv = data_folder / "selected_stations_090deg.csv"

# Time range
start_date = "2019-07-01"
end_date = "2019-07-08"

# Horizontal padding on both sides of the x-axis, unit: days
X_PAD_DAYS = 0.18

# ============================================================
# Gradient color settings
# ============================================================
# Use stronger colors to avoid gradients that are too light to distinguish
DE_BASE_COLOR = "#ff0a54"     # Dark red
DE_LIGHT_COLOR = "#f7cad0"    # Light red, but not close to white

DN_BASE_COLOR = "#0d47a1"     # Dark blue
DN_LIGHT_COLOR = "#bbdefb"    # Light blue, but not close to white

# Display only these dates on the x-axis
XTICK_DATES = pd.to_datetime([
    "2019-07-02",
    "2019-07-04",
    "2019-07-06",
    "2019-07-08",
])

XTICK_LABELS = ["02", "04", "06", "08"]

# Whether to subtract the first-day value of each station
zero_by_first_day = False


# ============================================================
# 2. Plotting style
# ============================================================

def set_plot_style():
    plt.rcParams["font.family"] = "Arial"
    plt.rcParams["font.size"] = 8
    plt.rcParams["axes.labelsize"] = 11
    plt.rcParams["xtick.labelsize"] = 10
    plt.rcParams["ytick.labelsize"] = 10
    plt.rcParams["legend.fontsize"] = 7
    plt.rcParams["axes.linewidth"] = 0.8
    plt.rcParams["xtick.major.width"] = 0.8
    plt.rcParams["ytick.major.width"] = 0.8
    plt.rcParams["xtick.minor.width"] = 0.6
    plt.rcParams["ytick.minor.width"] = 0.6
    plt.rcParams["xtick.major.size"] = 4.0
    plt.rcParams["ytick.major.size"] = 4.0
    plt.rcParams["xtick.minor.size"] = 2.2
    plt.rcParams["ytick.minor.size"] = 2.2
    plt.rcParams["pdf.fonttype"] = 42
    plt.rcParams["ps.fonttype"] = 42
    plt.rcParams["figure.facecolor"] = "white"
    plt.rcParams["axes.facecolor"] = "white"
    plt.rcParams["savefig.facecolor"] = "white"


# ============================================================
# 3. Read data
# ============================================================

def read_selected_stations(selected_csv):
    """
    Read the generated selected_stations_xxxdeg.csv.
    """
    if not selected_csv.exists():
        raise FileNotFoundError(f"Station-selection file not found: {selected_csv}")

    df = pd.read_csv(selected_csv)

    required_cols = ["station"]
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"{selected_csv.name} is missing required columns: {missing_cols}")

    return df


def find_station_csv(row):
    """
    Preferentially use csv_path from the selected_stations file.
    If csv_path does not exist, try to find station.csv in data_folder.
    """
    station = row["station"]

    if "csv_path" in row and pd.notna(row["csv_path"]):
        csv_path = Path(str(row["csv_path"]))
        if csv_path.exists():
            return csv_path

    fallback_csv = data_folder / f"{station}.csv"
    if fallback_csv.exists():
        return fallback_csv

    raise FileNotFoundError(
        f"Cannot find the original time-series CSV for station {station}."
        f"Please check csv_path in the selected_stations file, or confirm whether {fallback_csv} exists."
    )


def load_station_timeseries(csv_path):
    """
    Read dE/dN for a single station within the specified time period.
    """
    df = pd.read_csv(csv_path)

    required_cols = ["YYYYMMDD", "dE", "dN"]
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"{csv_path.name} is missing required columns: {missing_cols}")

    df["YYYYMMDD"] = pd.to_datetime(df["YYYYMMDD"], format="%Y%m%d")
    df = df.set_index("YYYYMMDD")
    df = df.loc[start_date:end_date].copy()

    return df


# ============================================================
# 4. Gradient colors and colorbar
# ============================================================

def make_gradient_info(selected_df, base_color, light_color, cmap_name):
    """
    Generate a more distinct color gradient.

    If center_distance_km is available in selected_df:
        1. Sort by distance from far to near;
        2. Plot farther stations first using lighter colors;
        3. Plot nearer stations later using darker colors;
        4. Assign colors at equal intervals according to the number of stations,
           avoiding indistinct colors when actual distances are too concentrated.

    If center_distance_km is unavailable:
        Apply a gradient according to station order.
    """

    cmap = LinearSegmentedColormap.from_list(
        cmap_name,
        [base_color, light_color],
        N=256
    )

    selected_df = selected_df.copy()

    if "center_distance_km" in selected_df.columns:
        selected_df["center_distance_km"] = selected_df["center_distance_km"].astype(float)

        # Key: farther stations first with lighter colors; nearer stations later with darker colors
        selected_df = selected_df.sort_values(
            "center_distance_km",
            ascending=False
        ).reset_index(drop=True)

        distance_values = selected_df["center_distance_km"].values

        if distance_values.max() == distance_values.min():
            norm = Normalize(
                vmin=distance_values.min() - 0.5,
                vmax=distance_values.max() + 0.5
            )
        else:
            norm = Normalize(
                vmin=distance_values.min(),
                vmax=distance_values.max()
            )

        # Farther distance: lighter color; nearer distance: darker color
        # Since selected_df is ordered from far to near, use positions from 1 to 0
        color_positions = np.linspace(1.0, 0.0, len(selected_df))
        colors = [cmap(p) for p in color_positions]

        cbar_label = "Distance to center (km)"
        has_distance = True

    else:
        selected_df = selected_df.reset_index(drop=True)

        values = np.arange(len(selected_df))
        norm = Normalize(vmin=0, vmax=max(len(selected_df) - 1, 1))

        # Without distance information, apply colors from dark to light according to station order
        color_positions = np.linspace(0.0, 1.0, len(selected_df))
        colors = [cmap(p) for p in color_positions]

        cbar_label = "Station order"
        has_distance = False

    return {
        "selected_df": selected_df,
        "colors": colors,
        "cmap": cmap,
        "norm": norm,
        "cbar_label": cbar_label,
        "has_distance": has_distance
    }


def add_colorbar(fig, ax, cmap, norm, label):
    """
    Add a colorbar to a single subplot.
    """
    sm = ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        ax=ax,
        pad=0.018,
        fraction=0.035
    )

    cbar.set_label(label, labelpad=5, fontsize=8)

    cbar.ax.tick_params(
        direction="in",
        length=3,
        width=0.7,
        labelsize=7
    )

    cbar.ax.yaxis.set_major_locator(MaxNLocator(nbins=4))

    # Display shorter distances and darker colors at the top
    cbar.ax.invert_yaxis()

    for spine in cbar.ax.spines.values():
        spine.set_linewidth(0.7)
        spine.set_color("black")


# ============================================================
# 5. Plotting function for a single panel
# ============================================================

def plot_component_panel(ax, selected_df, component, colors, ylabel, yticks, yticklabels, ylim, show_xlabel=False, show_xticklabels=False):
    """
    component: "dE" or "dN"
    The legend displays station names only.
    """
    for color, (_, row) in zip(colors, selected_df.iterrows()):
        station = row["station"]
        csv_path = find_station_csv(row)

        df = load_station_timeseries(csv_path)

        if df.empty:
            print(f"Skipping {station}: no data from {start_date} to {end_date}")
            continue

        y = df[component].copy()

        if zero_by_first_day:
            y = y - y.iloc[0]

        ax.plot(
            df.index,
            y,
            marker="o",
            markersize=4.8,        # Enlarge markers
            linewidth=1.9,         # Thicken lines to enhance gradient visibility
            color=color,
            markerfacecolor=color,
            markeredgecolor=color,
            markeredgewidth=0.3,
            alpha=0.98,
            label=station
        )

    # Remove title
    ax.set_title("")

    # Y-axis
    ax.set_ylabel(ylabel)

    # Fix the y-axis range and displayed values
    ax.set_ylim(ylim)
    ax.set_yticks(yticks)
    ax.set_yticklabels(yticklabels)
    ax.yaxis.set_minor_locator(AutoMinorLocator(4))

    # X-axis range and ticks: leave a small margin on both sides
    x_min = pd.to_datetime(start_date) - pd.Timedelta(days=X_PAD_DAYS)
    x_max = pd.to_datetime(end_date) + pd.Timedelta(days=X_PAD_DAYS)

    ax.set_xlim(x_min, x_max)
    ax.set_xticks(XTICK_DATES)
    ax.set_xticklabels(XTICK_LABELS)

    # Minor ticks: one minor tick per day
    ax.xaxis.set_minor_locator(mdates.DayLocator(interval=1))

    if show_xlabel:
        ax.set_xlabel("Date (2019-07)")
    else:
        ax.set_xlabel("")

    if not show_xticklabels:
        ax.tick_params(axis="x", labelbottom=False)

    # Display ticks only on the left and bottom
    ax.tick_params(
        axis="both",
        which="major",
        direction="in",
        length=4.0,
        width=0.8,
        top=False,
        bottom=True,
        left=True,
        right=False,
        labeltop=False,
        labelright=False,
        color="black",
        labelcolor="black"
    )

    ax.tick_params(
        axis="both",
        which="minor",
        direction="in",
        length=2.2,
        width=0.6,
        top=False,
        bottom=True,
        left=True,
        right=False,
        color="black"
    )

    # Keep borders on all four sides
    for side in ["left", "right", "top", "bottom"]:
        ax.spines[side].set_visible(True)
        ax.spines[side].set_linewidth(0.8)
        ax.spines[side].set_color("black")

    ax.grid(False)

    # Legend displays station names only
    ax.legend(
        loc="best",
        frameon=False,
        fontsize=7,
        handlelength=2.0,
        borderpad=0.2,
        labelspacing=0.3
    )


# ============================================================
# 6. Main function
# ============================================================

def main():
    set_plot_style()

    output_png.parent.mkdir(parents=True, exist_ok=True)
    output_pdf.parent.mkdir(parents=True, exist_ok=True)

    selected_dE_df = read_selected_stations(selected_dE_csv)
    selected_dN_df = read_selected_stations(selected_dN_csv)

    # Gradient color information
    de_gradient = make_gradient_info(
        selected_dE_df,
        base_color=DE_BASE_COLOR,
        light_color=DE_LIGHT_COLOR,
        cmap_name="dE_gradient"
    )

    dn_gradient = make_gradient_info(
        selected_dN_df,
        base_color=DN_BASE_COLOR,
        light_color=DN_LIGHT_COLOR,
        cmap_name="dN_gradient"
    )

    fig, axes = plt.subplots(
        2,
        1,
        figsize=(7.2, 4.8),
        sharex=True,
        dpi=300
    )

    fig.patch.set_facecolor("white")

    # -----------------------------
    # Upper panel: dE, angle = 150°
    # -----------------------------
    plot_component_panel(
        ax=axes[0],
        selected_df=de_gradient["selected_df"],
        component="dE",
        colors=de_gradient["colors"],
        ylabel="dE displacement (m)",
        yticks=[-0.30, -0.25, -0.20, -0.15],
        yticklabels=["-0.30", "-0.25", "-0.20", "-0.15"],
        ylim=(-0.31, -0.125),
        show_xlabel=False,
        show_xticklabels=False
    )

    add_colorbar(
        fig=fig,
        ax=axes[0],
        cmap=de_gradient["cmap"],
        norm=de_gradient["norm"],
        label=de_gradient["cbar_label"]
    )

    # -----------------------------
    # Lower panel: dN, angle = 90°
    # -----------------------------
    plot_component_panel(
        ax=axes[1],
        selected_df=dn_gradient["selected_df"],
        component="dN",
        colors=dn_gradient["colors"],
        ylabel="dN displacement (m)",
        yticks=[-0.18, -0.14, -0.10, -0.06],
        yticklabels=["-0.18", "-0.14", "-0.10", "-0.06"],
        ylim=(-0.205, -0.04),
        show_xlabel=True,
        show_xticklabels=True
    )

    add_colorbar(
        fig=fig,
        ax=axes[1],
        cmap=dn_gradient["cmap"],
        norm=dn_gradient["norm"],
        label=dn_gradient["cbar_label"]
    )

    # Keep only one x-axis:
    # Do not display x-axis ticks and labels on the upper panel
    axes[0].tick_params(
        axis="x",
        which="both",
        bottom=False,
        labelbottom=False
    )

    # Display the x-axis on the lower panel
    axes[1].tick_params(
        axis="x",
        which="both",
        bottom=True,
        labelbottom=True
    )

    plt.subplots_adjust(
        left=0.13,
        right=0.90,
        bottom=0.12,
        top=0.98,
        hspace=0.06
    )

    fig.savefig(output_png, dpi=600, bbox_inches="tight", facecolor="white")
    fig.savefig(output_pdf, bbox_inches="tight", facecolor="white")
    plt.close(fig)

    print("Plotting completed. Files saved to:")
    print(output_png)
    print(output_pdf)


if __name__ == "__main__":
    main()
'''

Fig-S9

In [ ]:
#The data of the b value
'''
# -*- coding: utf-8 -*-

import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import AutoMinorLocator, FixedLocator
from matplotlib.patches import Rectangle


# ============================================================
# 0. Path settings
# ============================================================
PHI_EXCEL_PATH = r"D:\a\master\Earthquake-US\Fig-Use-2\Fig-3\IMS_phi_values_data.xlsx"

# New data: merged USGS data with a minimum magnitude of 1
QUAKE_CSV_PATH = r"D:\a\master\Earthquake-US\Fig-Use-2\Fig-new\Magnitude_threshold_compare\usgs_quakes_M1_20180706_20200501_merged.csv"

OUTPUT_DIR = r"D:\a\master\Earthquake-US\Fig-Use-2\Fig-new\Magnitude_threshold_compare"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# 1. Time range settings
# ============================================================
PLOT_START = pd.to_datetime("2018-07-06")
PLOT_END = pd.to_datetime("2020-05-01")


# ============================================================
# 2. b-value calculation parameters
# ============================================================
TIME_COL = "time"
MAG_COL = "mag"

WINDOW_DAYS = 30
STEP_DAYS = 1

# Calculate these four minimum-magnitude thresholds separately
MIN_MAG_THRESHOLDS = [1.0, 1.5, 2.0, 2.5]

BIN_WIDTH = 0.1
MC_CORRECTION = 0.2
MIN_EVENTS_ABOVE_MC = 30

# True: estimate Mc separately for each 30-day window
# False: use a fixed Mc for the entire catalog
USE_WINDOW_MC = True


# ============================================================
# 3. Global plotting parameters: maintain Nature double-column style
# ============================================================
FIG_WIDTH = 8.8
FIG_HEIGHT = 3.4

plt.rcParams["text.color"] = "black"
plt.rcParams["axes.labelcolor"] = "black"
plt.rcParams["xtick.color"] = "black"
plt.rcParams["ytick.color"] = "black"

plt.rcParams["font.family"] = "Arial"
plt.rcParams["mathtext.fontset"] = "stix"
plt.rcParams["font.size"] = 12
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["xtick.labelsize"] = 11
plt.rcParams["ytick.labelsize"] = 11
plt.rcParams["legend.fontsize"] = 11

plt.rcParams["axes.linewidth"] = 0.7
plt.rcParams["xtick.major.width"] = 0.7
plt.rcParams["ytick.major.width"] = 0.7
plt.rcParams["xtick.minor.width"] = 0.5
plt.rcParams["ytick.minor.width"] = 0.5

plt.rcParams["xtick.major.size"] = 3.5
plt.rcParams["ytick.major.size"] = 3.5
plt.rcParams["xtick.minor.size"] = 2.0
plt.rcParams["ytick.minor.size"] = 2.0

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["savefig.facecolor"] = "white"
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"] = "white"


# ============================================================
# 4. Read Phi data
# ============================================================
def load_phi_data(excel_path):
    df = pd.read_excel(excel_path)

    df["date"] = pd.to_datetime(df["date"])
    df["global_phi"] = pd.to_numeric(df["global_phi_102"], errors="coerce")
    df["average_phi"] = pd.to_numeric(df["average_phi_102"], errors="coerce")
    df["diff_phi"] = pd.to_numeric(df["diff_phi_102"], errors="coerce")

    df = df[["date", "global_phi", "average_phi", "diff_phi"]].dropna()
    df = df.sort_values("date").reset_index(drop=True)

    return df


# ============================================================
# 5. Mc estimation: maximum curvature + correction
# ============================================================
def estimate_mc_max_curvature(magnitudes, bin_width=0.1, correction=0.2, min_mc=1.0):
    mags = pd.Series(magnitudes).dropna().astype(float)

    if len(mags) == 0:
        return np.nan

    binned = np.floor((mags + 1e-9) / bin_width) * bin_width
    binned = np.round(binned, 4)

    counts = pd.Series(binned).value_counts().sort_index()

    if counts.empty:
        return np.nan

    max_count = counts.max()
    peak_mag = counts[counts == max_count].index.min()

    mc = peak_mag + correction
    mc = max(mc, min_mc)

    mc = round(mc / bin_width) * bin_width

    return float(round(mc, 3))


# ============================================================
# 6. b-value calculation: maximum-likelihood method
# ============================================================
def calculate_b_value(magnitudes, mc, bin_width=0.1, min_events=30):
    mags = pd.Series(magnitudes).dropna().astype(float)
    mags = mags[mags >= mc]

    n = len(mags)

    if n < min_events:
        return np.nan, np.nan, n, np.nan

    mean_mag = mags.mean()
    denominator = mean_mag - (mc - bin_width / 2.0)

    if denominator <= 0:
        return np.nan, mean_mag, n, np.nan

    b_value = math.log10(math.e) / denominator

    if n > 1:
        sigma_b = 2.30 * (b_value ** 2) * np.sqrt(
            np.sum((mags - mean_mag) ** 2) / (n * (n - 1))
        )
    else:
        sigma_b = np.nan

    return b_value, mean_mag, n, sigma_b


# ============================================================
# 7. Read earthquake data
# ============================================================
def load_quake_data(csv_path):
    df = pd.read_csv(csv_path)

    if TIME_COL not in df.columns:
        raise ValueError(f"Time column not found: {TIME_COL}")

    if MAG_COL not in df.columns:
        raise ValueError(f"Magnitude column not found: {MAG_COL}")

    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce", utc=True)
    df[MAG_COL] = pd.to_numeric(df[MAG_COL], errors="coerce")

    df = df.dropna(subset=[TIME_COL, MAG_COL])

    if "type" in df.columns:
        df = df[df["type"].astype(str).str.lower() == "earthquake"]

    df = df.sort_values(TIME_COL).reset_index(drop=True)

    return df


# ============================================================
# 8. Calculate the 30-day rolling b-value for a given minimum-magnitude threshold
# ============================================================
def calculate_rolling_b_value(df_quake_all, min_mag_threshold):
    df = df_quake_all[df_quake_all[MAG_COL] >= min_mag_threshold].copy()
    df = df.sort_values(TIME_COL).reset_index(drop=True)

    if df.empty:
        raise ValueError(f"No available earthquake data after applying M >= {min_mag_threshold}.")

    global_mc = estimate_mc_max_curvature(
        df[MAG_COL],
        bin_width=BIN_WIDTH,
        correction=MC_CORRECTION,
        min_mc=min_mag_threshold
    )

    print(f"M >= {min_mag_threshold}: Estimated Mc for the entire dataset = {global_mc}")

    # Generate windows using a fixed time range to ensure identical time axes for all four figures
    start_day = PLOT_START
    end_day = PLOT_END

    window_end_days = pd.date_range(
        start=start_day + pd.Timedelta(days=WINDOW_DAYS - 1),
        end=end_day,
        freq=f"{STEP_DAYS}D"
    )

    results = []

    for window_end in window_end_days:
        window_start = window_end - pd.Timedelta(days=WINDOW_DAYS - 1)
        window_next_day = window_end + pd.Timedelta(days=1)

        # Note that df time is in UTC, so convert the window times to UTC as well
        window_start_utc = pd.to_datetime(window_start, utc=True)
        window_next_day_utc = pd.to_datetime(window_next_day, utc=True)

        window_df = df[
            (df[TIME_COL] >= window_start_utc) &
            (df[TIME_COL] < window_next_day_utc)
        ]

        mags_all = window_df[MAG_COL].values
        n_all = len(mags_all)

        if n_all == 0:
            results.append({
                "window_start": window_start.date(),
                "window_end": window_end,
                "date": window_end,
                "min_mag_threshold": min_mag_threshold,
                "n_all": 0,
                "mc": np.nan,
                "n_ge_mc": 0,
                "mean_mag_all": np.nan,
                "mean_mag_ge_mc": np.nan,
                "b_value": np.nan,
                "b_sigma": np.nan
            })
            continue

        mean_mag_all = np.mean(mags_all)

        if USE_WINDOW_MC:
            mc = estimate_mc_max_curvature(
                mags_all,
                bin_width=BIN_WIDTH,
                correction=MC_CORRECTION,
                min_mc=min_mag_threshold
            )
        else:
            mc = global_mc

        b_value, mean_mag_ge_mc, n_ge_mc, b_sigma = calculate_b_value(
            mags_all,
            mc=mc,
            bin_width=BIN_WIDTH,
            min_events=MIN_EVENTS_ABOVE_MC
        )

        results.append({
            "window_start": window_start.date(),
            "window_end": window_end,
            "date": window_end,
            "min_mag_threshold": min_mag_threshold,
            "n_all": n_all,
            "mc": mc,
            "n_ge_mc": n_ge_mc,
            "mean_mag_all": mean_mag_all,
            "mean_mag_ge_mc": mean_mag_ge_mc,
            "b_value": b_value,
            "b_sigma": b_sigma
        })

    res = pd.DataFrame(results)
    res["date"] = pd.to_datetime(res["date"])
    res = res.sort_values("date").reset_index(drop=True)

    return res


# ============================================================
# 9. Draw dashed boxes: keep unchanged
# ============================================================
def add_custom_box(ax, start_date, end_date, y_bottom, y_top, edgecolor="#99582a", linewidth=1.3, linestyle=(0, (7, 3.5))):
    start_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)

    x0 = mdates.date2num(start_date)
    x1 = mdates.date2num(end_date)

    rect = Rectangle(
        (x0, y_bottom),
        x1 - x0,
        y_top - y_bottom,
        fill=False,
        edgecolor=edgecolor,
        linewidth=linewidth,
        linestyle=linestyle,
        zorder=2
    )

    ax.add_patch(rect)


# ============================================================
# 10. Combined plot: three Phi curves + b-value
# ============================================================
def plot_phi_and_b_value(df_phi, df_b, min_mag_threshold, output_dir):
    df_phi_plot = df_phi[
        (df_phi["date"] >= PLOT_START) &
        (df_phi["date"] <= PLOT_END)
    ].copy()

    df_b_plot = df_b[
        (df_b["date"] >= PLOT_START) &
        (df_b["date"] <= PLOT_END)
    ].copy()

    if df_phi_plot.empty:
        raise ValueError("No Phi data are available within the specified time period.")

    if df_b_plot.empty:
        raise ValueError("No b-value data are available within the specified time period.")

    fig, ax_phi = plt.subplots(figsize=(FIG_WIDTH, FIG_HEIGHT))

    # Phi colors: keep unchanged
    color_global = "#1f4e79"
    color_avg = "#ca6702"
    color_diff = "#2e8b57"

    # Box colors: keep unchanged
    box1_color = "#ffbe0b"
    box2_color = "#fb5607"
    box3_color = "#8338ec"
    box4_color = "#3a86ff"

    # b-value color
    color_b = "tab:red"

    # Left axis: three Phi curves
    l1, = ax_phi.plot(
        df_phi_plot["date"],
        df_phi_plot["global_phi"],
        color=color_global,
        linewidth=1.55,
        label=r"$\Phi_{\mathrm{global}}$",
        zorder=4
    )

    l2, = ax_phi.plot(
        df_phi_plot["date"],
        df_phi_plot["average_phi"],
        color=color_avg,
        linewidth=1.50,
        label=r"$\overline{\Phi}_{\mathrm{regional}}$",
        zorder=4
    )

    l3, = ax_phi.plot(
        df_phi_plot["date"],
        df_phi_plot["diff_phi"],
        color=color_diff,
        linewidth=1.40,
        label=r"$\Delta \Phi_{\mathrm{rg}}$",
        zorder=4
    )

    ax_phi.set_xlabel("Date")
    ax_phi.set_ylabel(r"$\Phi$")

    # Right axis: b-value
    ax_b = ax_phi.twinx()

    l4, = ax_b.plot(
        df_b_plot["date"],
        df_b_plot["b_value"],
        color=color_b,
        linewidth=1.80,
        label=fr"$b$-value, $M \geq {min_mag_threshold:g}$",
        zorder=5
    )

    ax_b.set_ylabel("b-value", color="black")
    ax_b.tick_params(axis="y", which="major", direction="in", right=True, colors="black")
    ax_b.tick_params(axis="y", which="minor", direction="in", right=True, colors="black")
    ax_b.yaxis.set_minor_locator(AutoMinorLocator(5))

    # X-axis range
    ax_phi.set_xlim(PLOT_START, PLOT_END)

    major_tick_dates = pd.to_datetime([
        "2018-08-01",
        "2018-12-01",
        "2019-04-01",
        "2019-08-01",
        "2019-12-01",
        "2020-04-01"
    ])

    ax_phi.xaxis.set_major_locator(FixedLocator(mdates.date2num(major_tick_dates)))
    ax_phi.xaxis.set_minor_locator(mdates.MonthLocator(interval=1))
    ax_phi.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

    # Left-axis Phi range to ensure the boxes are fully displayed
    y_min = min(
        df_phi_plot["global_phi"].min(),
        df_phi_plot["average_phi"].min(),
        df_phi_plot["diff_phi"].min(),
        -0.10
    )
    y_max = max(
        df_phi_plot["global_phi"].max(),
        df_phi_plot["average_phi"].max(),
        df_phi_plot["diff_phi"].max(),
        0.97
    )

    y_range = y_max - y_min
    ax_phi.set_ylim(y_min - 0.04 * y_range, y_max + 0.04 * y_range)

    # Right-axis b-value range
    b_valid = df_b_plot["b_value"].dropna()

    if len(b_valid) > 0:
        b_min = b_valid.min()
        b_max = b_valid.max()
        b_range = b_max - b_min

        if b_range == 0:
            b_range = 0.1

        ax_b.set_ylim(b_min - 0.08 * b_range, b_max + 0.08 * b_range)

    # Tick style
    ax_phi.yaxis.set_minor_locator(AutoMinorLocator(5))

    ax_phi.tick_params(
        axis="both",
        which="major",
        direction="in",
        bottom=True,
        left=True,
        top=False,
        right=False
    )

    ax_phi.tick_params(
        axis="both",
        which="minor",
        direction="in",
        bottom=True,
        left=True,
        top=False,
        right=False
    )

    for label in ax_phi.get_xticklabels():
        label.set_ha("center")

    # Borders on all four sides
    for spine in ax_phi.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.7)
        spine.set_color("black")

    for spine in ax_b.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.7)
        spine.set_color("black")

    ax_phi.grid(False)
    ax_b.grid(False)
    ax_phi.margins(x=0.01)

    # Dashed boxes: keep unchanged
    add_custom_box(
        ax_phi,
        "2019-05-03",
        "2019-06-30",
        y_bottom=-0.10,
        y_top=0.97,
        edgecolor=box1_color,
        linewidth=1.3,
        linestyle=(0, (7, 3.5))
    )

    add_custom_box(
        ax_phi,
        "2019-07-03",
        "2019-08-16",
        y_bottom=-0.10,
        y_top=0.97,
        edgecolor=box2_color,
        linewidth=1.3,
        linestyle=(0, (7, 3.5))
    )

    add_custom_box(
        ax_phi,
        "2019-08-20",
        "2019-10-31",
        y_bottom=-0.10,
        y_top=0.97,
        edgecolor=box3_color,
        linewidth=1.3,
        linestyle=(0, (7, 3.5))
    )

    add_custom_box(
        ax_phi,
        "2019-05-25",
        "2019-06-18",
        y_bottom=-0.05,
        y_top=0.50,
        edgecolor=box4_color,
        linewidth=1.3,
        linestyle=(0, (7, 3.5))
    )

    # Legend
    handles = [l1, l2, l3, l4]
    labels = [h.get_label() for h in handles]

    ax_phi.legend(
        handles,
        labels,
        loc="lower left",
        bbox_to_anchor=(0.006, 0.18),
        frameon=False,
        handlelength=1.3,
        handletextpad=0.45,
        borderpad=0.12,
        labelspacing=0.22
    )

    plt.tight_layout(pad=0.45)

    # Save
    threshold_str = str(min_mag_threshold).replace(".", "p")

    png_path = os.path.join(
        output_dir,
        f"Phi_and_b_value_Mge{threshold_str}_20180706_20200501.png"
    )
    pdf_path = os.path.join(
        output_dir,
        f"Phi_and_b_value_Mge{threshold_str}_20180706_20200501.pdf"
    )

    plt.savefig(png_path, dpi=600, bbox_inches="tight")
    plt.savefig(pdf_path, bbox_inches="tight")
    plt.close()

    print(f"M >= {min_mag_threshold}: PNG saved to: {png_path}")
    print(f"M >= {min_mag_threshold}: PDF saved to: {pdf_path}")


# ============================================================
# 11. Main program
# ============================================================
if __name__ == "__main__":
    df_phi = load_phi_data(PHI_EXCEL_PATH)
    df_quake_all = load_quake_data(QUAKE_CSV_PATH)

    all_b_results = []

    for min_mag in MIN_MAG_THRESHOLDS:
        print("=" * 70)
        print(f"Starting b-value calculation for M >= {min_mag}")

        df_b = calculate_rolling_b_value(
            df_quake_all=df_quake_all,
            min_mag_threshold=min_mag
        )

        all_b_results.append(df_b)

        # Save the b-value result table for each threshold separately
        threshold_str = str(min_mag).replace(".", "p")
        b_csv_out = os.path.join(
            OUTPUT_DIR,
            f"rolling_30day_b_value_Mge{threshold_str}_20180706_20200501.csv"
        )
        df_b.to_csv(b_csv_out, index=False, encoding="utf-8-sig")
        print(f"M >= {min_mag}: b-value CSV saved to: {b_csv_out}")

        # Plot
        plot_phi_and_b_value(
            df_phi=df_phi,
            df_b=df_b,
            min_mag_threshold=min_mag,
            output_dir=OUTPUT_DIR
        )

    # Combine and save the b-value results for all four thresholds
    df_b_all = pd.concat(all_b_results, ignore_index=True)
    all_csv_out = os.path.join(
        OUTPUT_DIR,
        "rolling_30day_b_value_all_thresholds_20180706_20200501.csv"
    )
    df_b_all.to_csv(all_csv_out, index=False, encoding="utf-8-sig")

    print("=" * 70)
    print(f"Combined b-value results for all four thresholds saved to: {all_csv_out}")
    print("All completed.")
'''

In [ ]:
# (a) Count and b-value plot
'''
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import NullFormatter, AutoMinorLocator
from pathlib import Path


# ============================================================
# 0. Configuration: mainly modify this section in the future
# ============================================================

# ---------- Plotting time range ----------
PLOT_START_DATE = pd.Timestamp("2019-05-06")
PLOT_END_DATE = pd.Timestamp("2019-11-06")

# ---------- b-value switching date ----------
SPLIT_DATE = pd.Timestamp("2019-07-04")

# ---------- Mainshock date ----------
MAIN_DATE = pd.Timestamp("2019-07-06")

# ---------- Shaded intervals ----------
SHADE_1_START = pd.Timestamp("2019-05-25")
SHADE_1_END = pd.Timestamp("2019-06-18")

SHADE_2_START = pd.Timestamp("2019-07-03")
SHADE_2_END = pd.Timestamp("2019-08-08")

# ---------- X-axis display ----------
XTICK_DATES = pd.to_datetime([
    "2019-06-01",
    "2019-07-01",
    "2019-08-01",
    "2019-09-01",
    "2019-10-01",
])

XTICK_LABELS = [
    "2019-06",
    "2019-07",
    "2019-08",
    "2019-09",
    "2019-10",
]

# ---------- Input files ----------
QUAKE_FILE = Path(
    r"D:\a\master\Earthquake-US\Fig-Use-2\Fig-new\Magnitude_threshold_compare\usgs_quakes_M1_20180706_20200501_merged.csv"
)

B_FILE_MGE1 = Path(
    r"D:\a\master\Earthquake-US\Fig-Use-2\Fig-new\Magnitude_threshold_compare\rolling_30day_b_value_Mge1p0_20180706_20200501.csv"
)

B_FILE_MGE2 = Path(
    r"D:\a\master\Earthquake-US\Fig-Use-2\Fig-new\Magnitude_threshold_compare\rolling_30day_b_value_Mge2p0_20180706_20200501.csv"
)

# ---------- Output directory ----------
OUT_DIR = Path(
    r"D:\a\master\Earthquake-US\Fig-Over-Output\Fig-S9"
)

# ---------- Whether to display the figure ----------
SHOW_FIGURE = False

# ---------- Whether to save the figure ----------
SAVE_FIGURE = True

# ---------- Figure size ----------
FIG_SIZE = (11, 5.8)


# ============================================================
# 1. Plotting style function
# ============================================================

def setup_plot_style():
    """
    Set the overall plotting style.
    """
    plt.rcParams["font.family"] = "Arial"

    # Font sizes
    plt.rcParams["font.size"] = 14
    plt.rcParams["axes.labelsize"] = 17
    plt.rcParams["xtick.labelsize"] = 14
    plt.rcParams["ytick.labelsize"] = 14
    plt.rcParams["legend.fontsize"] = 13

    plt.rcParams["axes.linewidth"] = 1.1
    plt.rcParams["axes.edgecolor"] = "black"
    plt.rcParams["axes.grid"] = False

    plt.rcParams["xtick.major.width"] = 1.0
    plt.rcParams["ytick.major.width"] = 1.0
    plt.rcParams["xtick.minor.width"] = 0.7
    plt.rcParams["ytick.minor.width"] = 0.7

    plt.rcParams["xtick.major.size"] = 5.5
    plt.rcParams["ytick.major.size"] = 4.5
    plt.rcParams["xtick.minor.size"] = 2.6
    plt.rcParams["ytick.minor.size"] = 2.4

    plt.rcParams["pdf.fonttype"] = 42
    plt.rcParams["ps.fonttype"] = 42

    plt.rcParams["savefig.facecolor"] = "white"
    plt.rcParams["figure.facecolor"] = "white"
    plt.rcParams["axes.facecolor"] = "white"

    # Control the linewidth of hatch shading
    plt.rcParams["hatch.linewidth"] = 0.45


def strengthen_axes_frame(ax1, ax2):
    ax2.patch.set_visible(False)

    ax1.set_frame_on(True)
    ax2.set_frame_on(True)

    for side in ["left", "bottom", "top"]:
        ax1.spines[side].set_visible(True)
        ax1.spines[side].set_color("black")
        ax1.spines[side].set_linewidth(1.1)

    ax1.spines["right"].set_visible(False)

    ax2.spines["right"].set_visible(True)
    ax2.spines["right"].set_color("black")
    ax2.spines["right"].set_linewidth(1.1)
    ax2.spines["right"].set_position(("axes", 1.0))

    for side in ["left", "bottom", "top"]:
        ax2.spines[side].set_visible(False)

    ax1.set_axisbelow(True)
    ax2.set_axisbelow(True)


def configure_time_axis(ax, start_date, end_date):
    ax.set_xlim(start_date, end_date)

    ax.set_xticks(XTICK_DATES)
    ax.set_xticklabels(XTICK_LABELS)

    # Minor ticks: place them within each month to avoid overlap with major ticks at the beginning of the month
    ax.xaxis.set_minor_locator(
        mdates.MonthLocator(bymonthday=[1, 6, 12, 18, 24])
    )

    ax.xaxis.set_minor_formatter(
        NullFormatter()
    )

    for label in ax.get_xticklabels():
        label.set_rotation(0)
        label.set_ha("center")


def configure_ticks(ax1, ax2):
    """
    Display ticks only on the left, bottom, and right sides;
    do not display top ticks.
    """
    ax1.tick_params(
        axis="x",
        which="major",
        direction="in",
        bottom=True,
        top=False,
        labelbottom=True,
        labeltop=False,
        length=5.5,
        width=1.0,
        pad=6
    )

    ax1.tick_params(
        axis="x",
        which="minor",
        direction="in",
        bottom=True,
        top=False,
        length=2.6,
        width=0.7
    )

    ax1.tick_params(
        axis="y",
        which="major",
        direction="in",
        left=True,
        right=False,
        labelleft=True,
        labelright=False,
        length=4.5,
        width=1.0,
        pad=6
    )

    ax1.tick_params(
        axis="y",
        which="minor",
        direction="in",
        left=True,
        right=False,
        length=2.4,
        width=0.7
    )

    ax2.tick_params(
        axis="y",
        which="major",
        direction="in",
        right=True,
        left=False,
        labelright=True,
        labelleft=False,
        length=4.5,
        width=1.0,
        pad=6
    )

    ax2.tick_params(
        axis="y",
        which="minor",
        direction="in",
        right=True,
        left=False,
        length=2.4,
        width=0.7
    )


def configure_grid(ax1, ax2):
    ax1.grid(False)
    ax1.xaxis.grid(False, which="both")
    ax1.yaxis.grid(False, which="both")

    ax2.grid(False)
    ax2.xaxis.grid(False, which="both")
    ax2.yaxis.grid(False, which="both")


def add_shaded_intervals(ax):
    """
    Add two hatched shaded intervals.
    """

    # First interval: 2019-05-25 ~ 2019-06-18
    ax.axvspan(
        SHADE_1_START,
        SHADE_1_END,
        facecolor="none",
        edgecolor="#3a86ff",
        hatch="///",
        linewidth=0.0,
        zorder=0
    )

    # Second interval: 2019-07-03 ~ 2019-08-08
    ax.axvspan(
        SHADE_2_START,
        SHADE_2_END,
        facecolor="none",
        edgecolor="#fb5607",
        hatch="///",
        linewidth=0.0,
        zorder=0
    )


# ============================================================
# 2. Data reading functions
# ============================================================

def read_quake_data(quake_file):
    quakes = pd.read_csv(quake_file)

    if "time" not in quakes.columns:
        raise ValueError(f"Cannot find the time column in the earthquake file: {quake_file}")

    if "mag" not in quakes.columns:
        raise ValueError(f"Cannot find the mag column in the earthquake file: {quake_file}")

    quakes["time"] = pd.to_datetime(
        quakes["time"],
        utc=True,
        errors="coerce"
    )

    quakes["mag"] = pd.to_numeric(
        quakes["mag"],
        errors="coerce"
    )

    quakes = quakes.dropna(
        subset=["time", "mag"]
    ).copy()

    quakes["date"] = quakes["time"].dt.tz_convert(None).dt.floor("D")

    return quakes


def read_b_value_file(file_path, source_label):
    df = pd.read_csv(file_path)

    for col in ["window_start", "window_end", "date"]:
        if col in df.columns:
            df[col] = pd.to_datetime(
                df[col],
                errors="coerce"
            )

    if "date" not in df.columns:
        if "window_end" in df.columns:
            df["date"] = df["window_end"]
        else:
            raise ValueError(
                f"{file_path} contains neither a date column nor a window_end column."
            )

    if "window_end" not in df.columns:
        df["window_end"] = df["date"]

    if "window_start" not in df.columns:
        df["window_start"] = df["window_end"] - pd.Timedelta(days=29)

    if "b_value" not in df.columns:
        raise ValueError(f"Cannot find the b_value column in {file_path}.")

    df["b_value"] = pd.to_numeric(
        df["b_value"],
        errors="coerce"
    )

    df = df.dropna(
        subset=[
            "window_start",
            "window_end",
            "date",
            "b_value"
        ]
    ).copy()

    df["b_source"] = source_label

    df = df[
        [
            "date",
            "window_start",
            "window_end",
            "b_value",
            "b_source"
        ]
    ].copy()

    df = df.sort_values("date").reset_index(drop=True)

    return df


# ============================================================
# 3. b-value splicing and time-filtering functions
# ============================================================

def build_piecewise_b_value_dataframe(b_file_mge1, b_file_mge2, split_date):
    bdf_mge1 = read_b_value_file(
        b_file_mge1,
        source_label="b-value, M≥1.0"
    )

    bdf_mge2 = read_b_value_file(
        b_file_mge2,
        source_label="b-value, M≥2.0"
    )

    bdf_before = bdf_mge1[
        bdf_mge1["date"] < split_date
    ].copy()

    bdf_after = bdf_mge2[
        bdf_mge2["date"] >= split_date
    ].copy()

    bdf = pd.concat(
        [bdf_before, bdf_after],
        ignore_index=True
    )

    bdf = bdf.sort_values("date").reset_index(drop=True)

    bdf = bdf.drop_duplicates(
        subset=["date"],
        keep="last"
    ).reset_index(drop=True)

    return bdf


def filter_by_plot_date_range(df, start_date, end_date):
    df = df[
        (df["date"] >= start_date) &
        (df["date"] <= end_date)
    ].copy()

    df = df.sort_values("date").reset_index(drop=True)

    if df.empty:
        raise ValueError(
            f"No data are available within the current time range {start_date.date()} to {end_date.date()}."
        )

    return df


# ============================================================
# 4. Magnitude-bin counting function
# ============================================================

def compute_magnitude_bin_counts(quakes, bdf):
    mag_bins = {
        "1 ≤ M < 2": (1, 2),
        "2 ≤ M < 3": (2, 3),
        "3 ≤ M < 4": (3, 4),
        "M ≥ 4": (4, None),
    }

    result = bdf[
        [
            "date",
            "window_start",
            "window_end",
            "b_value",
            "b_source"
        ]
    ].copy()

    for label, (m_min, m_max) in mag_bins.items():
        counts = []

        for start, end in zip(result["window_start"], result["window_end"]):
            time_mask = (
                (quakes["date"] >= start) &
                (quakes["date"] <= end)
            )

            if m_max is None:
                mag_mask = quakes["mag"] >= m_min
            else:
                mag_mask = (
                    (quakes["mag"] >= m_min) &
                    (quakes["mag"] < m_max)
                )

            mask = time_mask & mag_mask
            counts.append(int(mask.sum()))

        result[f"N_{label}"] = counts

    return result, mag_bins


# ============================================================
# 5. Output path function
# ============================================================

def build_output_paths(out_dir, start_date, end_date):
    out_dir.mkdir(parents=True, exist_ok=True)

    date_tag = (
        f"{start_date.strftime('%Y%m%d')}_"
        f"{end_date.strftime('%Y%m%d')}"
    )

    out_png = out_dir / f"Count_Mbin1234_and_piecewise_bvalue_30day_{date_tag}.png"
    out_pdf = out_dir / f"Count_Mbin1234_and_piecewise_bvalue_30day_{date_tag}.pdf"
    out_csv = out_dir / f"Count_Mbin1234_and_piecewise_bvalue_30day_{date_tag}.csv"

    return out_png, out_pdf, out_csv


# ============================================================
# 6. Plotting function
# ============================================================

def plot_count_and_piecewise_b_value(result, mag_bins, split_date, start_date, end_date, out_png, out_pdf, save_figure=True, show_figure=True):
    setup_plot_style()

    fig, ax1 = plt.subplots(figsize=FIG_SIZE)

    # ----------------------
    # Add shaded intervals
    # ----------------------
    add_shaded_intervals(ax1)

    # ----------------------
    # Left axis: earthquake counts
    # ----------------------
    count_colors = {
        "1 ≤ M < 2": "#90e0ef",
        "2 ≤ M < 3": "#00b4d8",
        "3 ≤ M < 4": "#0077b6",
        "M ≥ 4": "#03045e",
    }

    for label in mag_bins.keys():
        y = result[f"N_{label}"].astype(float).replace(0, np.nan)

        ax1.plot(
            result["date"],
            y,
            color=count_colors[label],
            marker="o",
            markersize=3.2,
            linewidth=1.7,
            linestyle="-",
            label=label,
            zorder=4
        )

    ax1.set_yscale("log")
    ax1.set_ylabel("Count", fontsize=18, labelpad=8)
    ax1.set_xlabel("Date", fontsize=18, labelpad=8)

    # ----------------------
    # 2019-07-04 vertical line: retained
    # ----------------------
    if start_date <= SPLIT_DATE <= end_date:
        ax1.axvline(
            SPLIT_DATE,
            color="gray",
            linestyle="--",
            linewidth=1.3,
            alpha=0.90,
            label="Foreshock",
            zorder=3
        )

    # ----------------------
    # Mainshock vertical dashed line: 2019-07-06
    # ----------------------
    if start_date <= MAIN_DATE <= end_date:
        ax1.axvline(
            MAIN_DATE,
            color="black",
            linestyle="--",
            linewidth=1.4,
            alpha=0.85,
            label="Main",
            zorder=3
        )

    # ----------------------
    # Right axis: b-value
    # ----------------------
    ax2 = ax1.twinx()

    ax2.grid(False)
    ax2.yaxis.grid(False, which="both")
    ax2.xaxis.grid(False, which="both")
    ax2.patch.set_visible(False)

    b_before = result[
        result["date"] < split_date
    ].copy()

    b_after = result[
        result["date"] >= split_date
    ].copy()

    # Use clearly contrasting red colors for the two b-values
    b_color_mge1 = "#ff8fab"
    b_color_mge2 = "red"

    ax2.plot(
        b_before["date"],
        b_before["b_value"],
        color=b_color_mge1,
        marker="s",
        markersize=3.4,
        linewidth=2.1,
        linestyle="-",
        label="b, M≥1.0",
        zorder=5
    )

    ax2.plot(
        b_after["date"],
        b_after["b_value"],
        color=b_color_mge2,
        marker="s",
        markersize=3.4,
        linewidth=2.1,
        linestyle="-",
        label="b, M≥2.0",
        zorder=5
    )

    ax2.set_ylabel("b", fontsize=18, labelpad=8)

    # Display specified values on the right y-axis
    ax2.set_ylim(0.55, 1.25)
    ax2.set_yticks([0.6, 0.8, 1.0, 1.2])
    ax2.set_yticklabels(["0.6", "0.8", "1.0", "1.2"])

    # Display 4 minor ticks between two major ticks
    ax2.yaxis.set_minor_locator(
        AutoMinorLocator(5)
    )

    # ----------------------
    # Axes, ticks, grid, and borders
    # ----------------------
    configure_time_axis(
        ax=ax1,
        start_date=start_date,
        end_date=end_date
    )

    configure_ticks(ax1, ax2)

    configure_grid(ax1, ax2)

    strengthen_axes_frame(ax1, ax2)

    # ----------------------
    # Legend
    # ----------------------
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()

    ax1.legend(
        lines1 + lines2,
        labels1 + labels2,
        loc="upper left",
        frameon=False,
        fontsize=13,
        handlelength=2.2,
        labelspacing=0.45,
        borderpad=0.2
    )

    # Remove title
    ax1.set_title("")

    plt.tight_layout()

    if save_figure:
        plt.savefig(out_png, dpi=300, bbox_inches="tight")
        plt.savefig(out_pdf, bbox_inches="tight")

    if show_figure:
        plt.show()
    else:
        plt.close(fig)


# ============================================================
# 7. Main execution function
# ============================================================

def run_count_and_piecewise_b_value_plot():
    out_png, out_pdf, out_csv = build_output_paths(
        out_dir=OUT_DIR,
        start_date=PLOT_START_DATE,
        end_date=PLOT_END_DATE
    )

    quakes = read_quake_data(QUAKE_FILE)

    bdf = build_piecewise_b_value_dataframe(
        b_file_mge1=B_FILE_MGE1,
        b_file_mge2=B_FILE_MGE2,
        split_date=SPLIT_DATE
    )

    bdf = filter_by_plot_date_range(
        df=bdf,
        start_date=PLOT_START_DATE,
        end_date=PLOT_END_DATE
    )

    result, mag_bins = compute_magnitude_bin_counts(
        quakes=quakes,
        bdf=bdf
    )

    result.to_csv(
        out_csv,
        index=False,
        encoding="utf-8-sig"
    )

    plot_count_and_piecewise_b_value(
        result=result,
        mag_bins=mag_bins,
        split_date=SPLIT_DATE,
        start_date=PLOT_START_DATE,
        end_date=PLOT_END_DATE,
        out_png=out_png,
        out_pdf=out_pdf,
        save_figure=SAVE_FIGURE,
        show_figure=SHOW_FIGURE
    )

    print("\nAll completed.")
    print(f"Result CSV saved to: {out_csv}")
    print(f"PNG image saved to: {out_png}")
    print(f"PDF image saved to: {out_pdf}")


# ============================================================
# 8. Program entry point
# ============================================================

if __name__ == "__main__":
    run_count_and_piecewise_b_value_plot()
'''

Table-S1

In [ ]:
'''
import os
import glob
import pandas as pd


# ============================================================
# 1. Paths and 3×3 grid parameters
# ============================================================
CSV_DIR = r"D:\a\master\Earthquake-US\Data\20190706-102\PosData-7"
OUTPUT_DIR = r"D:\a\master\Earthquake-US\Fig-Over-Output\Site"

CENTER_LAT = 35.7695
CENTER_LON = 242.4006667
DELTA = 1.0


# ============================================================
# 2. Build the 3×3 regions exactly consistent with the original program
#
# i = 0, 1, 2: south, center, north
# j = 0, 1, 2: west, center, east
#
# The original program uses:
#   lower boundary <= coordinate < upper boundary
# ============================================================
def build_regions(center_lat, center_lon, delta):
    position_names = {
        (0, 0): "Southwest",
        (0, 1): "South",
        (0, 2): "Southeast",
        (1, 0): "West",
        (1, 1): "Center",
        (1, 2): "East",
        (2, 0): "Northwest",
        (2, 1): "North",
        (2, 2): "Northeast",
    }

    regions = {}

    for i in range(3):
        for j in range(3):
            lat_low = center_lat + (i - 1) * delta - delta / 2
            lat_high = center_lat + (i - 1) * delta + delta / 2

            lon_low = center_lon + (j - 1) * delta - delta / 2
            lon_high = center_lon + (j - 1) * delta + delta / 2

            region_name = f"region_{i}_{j}"

            regions[region_name] = {
                "i": i,
                "j": j,
                "position_cn": position_names[(i, j)],
                "lat_low": lat_low,
                "lat_high": lat_high,
                "lon_low": lon_low,
                "lon_high": lon_high,
            }

    return regions


# ============================================================
# 3. Determine which region a station belongs to based on latitude and longitude
# ============================================================
def find_region(lat, lon, regions):
    for region_name, region in regions.items():
        in_lat = region["lat_low"] <= lat < region["lat_high"]
        in_lon = region["lon_low"] <= lon < region["lon_high"]

        if in_lat and in_lon:
            return region_name

    return None


# ============================================================
# 4. Read station names and coordinates from each CSV file
#
# The filename (without .csv) is used as the station name.
# Only the NLat and Elong columns are read; the full displacement
# time series is not loaded.
# ============================================================
def read_station_information(csv_dir):
    csv_files = sorted(glob.glob(os.path.join(csv_dir, "*.csv")))

    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in directory: {csv_dir}")

    station_rows = []
    failed_files = []

    for file_path in csv_files:
        station_name = os.path.splitext(os.path.basename(file_path))[0]

        try:
            coord_df = pd.read_csv(
                file_path,
                usecols=["NLat", "Elong"]
            ).dropna(subset=["NLat", "Elong"])

            if coord_df.empty:
                failed_files.append({
                    "station_name": station_name,
                    "file_path": file_path,
                    "reason": "No valid values in NLat or Elong"
                })
                continue

            # GNSS station coordinates are usually constant throughout the file;
            # use the first valid coordinate pair
            lat = float(coord_df.iloc[0]["NLat"])
            lon_original = float(coord_df.iloc[0]["Elong"])

            # Compatible with -180~180 longitude; the original program uses 0~360 longitude
            lon = lon_original + 360.0 if lon_original < 0 else lon_original

            station_rows.append({
                "station_name": station_name,
                "latitude": lat,
                "longitude_original": lon_original,
                "longitude_0_360": lon,
                "file_path": file_path,
            })

        except Exception as exc:
            failed_files.append({
                "station_name": station_name,
                "file_path": file_path,
                "reason": str(exc)
            })

    return pd.DataFrame(station_rows), pd.DataFrame(failed_files)


# ============================================================
# 5. Divide regions and export results
# ============================================================
def export_region_station_names(
    csv_dir,
    output_dir,
    center_lat,
    center_lon,
    delta
):
    os.makedirs(output_dir, exist_ok=True)

    regions = build_regions(
        center_lat=center_lat,
        center_lon=center_lon,
        delta=delta
    )

    station_df, failed_df = read_station_information(csv_dir)

    if station_df.empty:
        raise ValueError("No station coordinates were successfully read.")

    # --------------------------------------------------------
    # Assign each station to a region
    # --------------------------------------------------------
    station_df["region_name"] = station_df.apply(
        lambda row: find_region(
            lat=row["latitude"],
            lon=row["longitude_0_360"],
            regions=regions
        ),
        axis=1
    )

    station_df["position_cn"] = station_df["region_name"].map(
        {
            region_name: region["position_cn"]
            for region_name, region in regions.items()
        }
    )

    station_df = station_df.sort_values(
        by=["region_name", "station_name"],
        na_position="last"
    ).reset_index(drop=True)

    # --------------------------------------------------------
    # Output 1: station-by-station details
    # --------------------------------------------------------
    station_detail_file = os.path.join(
        output_dir,
        "01_station_region_details.csv"
    )

    station_df.to_csv(
        station_detail_file,
        index=False,
        encoding="utf-8-sig"
    )

    # --------------------------------------------------------
    # Output 2: summary for each region
    # Keep all nine regions, even if a region contains no stations
    # --------------------------------------------------------
    summary_rows = []

    for region_name, region in regions.items():
        sites = (
            station_df.loc[
                station_df["region_name"] == region_name,
                "station_name"
            ]
            .dropna()
            .astype(str)
            .sort_values()
            .tolist()
        )

        summary_rows.append({
            "region_name": region_name,
            "position_cn": region["position_cn"],
            "i": region["i"],
            "j": region["j"],
            "lat_low": region["lat_low"],
            "lat_high": region["lat_high"],
            "lon_low": region["lon_low"],
            "lon_high": region["lon_high"],
            "station_count": len(sites),
            "station_names": ", ".join(sites),
        })

    summary_df = pd.DataFrame(summary_rows)

    summary_file = os.path.join(
        output_dir,
        "02_station_names_by_region_summary.csv"
    )

    summary_df.to_csv(
        summary_file,
        index=False,
        encoding="utf-8-sig"
    )

    # --------------------------------------------------------
    # Output 3: an independent TXT file for each region
    # --------------------------------------------------------
    txt_dir = os.path.join(output_dir, "03_individual_station_lists_by_region")
    os.makedirs(txt_dir, exist_ok=True)

    for _, row in summary_df.iterrows():
        region_name = row["region_name"]
        position_cn = row["position_cn"]

        sites = (
            station_df.loc[
                station_df["region_name"] == region_name,
                "station_name"
            ]
            .dropna()
            .astype(str)
            .sort_values()
            .tolist()
        )

        txt_file = os.path.join(
            txt_dir,
            f"{region_name}_{position_cn}.txt"
        )

        with open(txt_file, "w", encoding="utf-8-sig") as file:
            file.write(f"Region: {region_name} ({position_cn})\n")
            file.write(
                f"Latitude range: [{row['lat_low']:.7f}, "
                f"{row['lat_high']:.7f})\n"
            )
            file.write(
                f"Longitude range: [{row['lon_low']:.7f}, "
                f"{row['lon_high']:.7f})\n"
            )
            file.write(f"Number of stations: {len(sites)}\n\n")

            if sites:
                for site in sites:
                    file.write(f"{site}\n")
            else:
                file.write("No stations in this region.\n")

    # --------------------------------------------------------
    # Output 4: stations outside the study area
    # --------------------------------------------------------
    outside_df = station_df[
        station_df["region_name"].isna()
    ].copy()

    outside_file = os.path.join(
        output_dir,
        "04_stations_outside_3x3_grid.csv"
    )

    outside_df.to_csv(
        outside_file,
        index=False,
        encoding="utf-8-sig"
    )

    # --------------------------------------------------------
    # Output 5: files that failed to be read
    # --------------------------------------------------------
    failed_file = os.path.join(
        output_dir,
        "05_failed_files.csv"
    )

    failed_df.to_csv(
        failed_file,
        index=False,
        encoding="utf-8-sig"
    )

    # --------------------------------------------------------
    # Console display
    # --------------------------------------------------------
    print("\n" + "=" * 72)
    print("Station names corresponding to each region")
    print("=" * 72)

    for _, row in summary_df.iterrows():
        region_name = row["region_name"]
        position_cn = row["position_cn"]
        station_names = row["station_names"]

        print(
            f"\n{region_name} ({position_cn})"
            f": {int(row['station_count'])} stations"
        )

        if station_names:
            print(station_names)
        else:
            print("No stations")

    print("\n" + "=" * 72)
    print(f"Number of stations successfully read: {len(station_df)}")
    print(f"Number of stations within the 3×3 grid: {station_df['region_name'].notna().sum()}")
    print(f"Number of stations outside the 3×3 grid: {station_df['region_name'].isna().sum()}")
    print(f"Number of files that failed to be read: {len(failed_df)}")

    print("\nOutput files:")
    print(station_detail_file)
    print(summary_file)
    print(txt_dir)
    print(outside_file)
    print(failed_file)

    return station_df, summary_df, outside_df, failed_df


# ============================================================
# 6. Program entry point
# ============================================================
if __name__ == "__main__":
    export_region_station_names(
        csv_dir=CSV_DIR,
        output_dir=OUTPUT_DIR,
        center_lat=CENTER_LAT,
        center_lon=CENTER_LON,
        delta=DELTA
    )
'''

Fig-SV1

In [ ]:
# Three phi-value plots used for animation (with a moving black vertical line)
'''import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import AutoMinorLocator, FixedLocator
from matplotlib.patches import Rectangle

# ========================
# 0. Global plotting parameters (Nature double-column style)
# ========================
FIG_WIDTH = 8.8
FIG_HEIGHT = 3.0

plt.rcParams['font.family'] = 'Arial'
plt.rcParams['mathtext.fontset'] = 'stix'
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['xtick.labelsize'] = 11
plt.rcParams['ytick.labelsize'] = 11
plt.rcParams['legend.fontsize'] = 12

plt.rcParams['axes.linewidth'] = 0.7
plt.rcParams['xtick.major.width'] = 0.7
plt.rcParams['ytick.major.width'] = 0.7
plt.rcParams['xtick.minor.width'] = 0.5
plt.rcParams['ytick.minor.width'] = 0.5

plt.rcParams['xtick.major.size'] = 3.5
plt.rcParams['ytick.major.size'] = 3.5
plt.rcParams['xtick.minor.size'] = 2.0
plt.rcParams['ytick.minor.size'] = 2.0

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['savefig.facecolor'] = 'white'
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'


# ========================
# 1. Read Excel data
# ========================
def load_phi_data(excel_path):
    df = pd.read_excel(excel_path)

    df['date'] = pd.to_datetime(df['date'])
    df['global_phi'] = pd.to_numeric(df['global_phi_102'], errors='coerce')
    df['average_phi'] = pd.to_numeric(df['average_phi_102'], errors='coerce')
    df['diff_phi'] = pd.to_numeric(df['diff_phi_102'], errors='coerce')

    df = df[['date', 'global_phi', 'average_phi', 'diff_phi']].dropna()
    df = df.sort_values('date').reset_index(drop=True)

    return df


# ========================
# 2. Draw dashed boxes
# ========================
def add_custom_box(ax, df, start_date, end_date, y_bottom=None, y_top=None, line_cols=('global_phi', 'average_phi', 'diff_phi'),
                   auto_pad_ratio=0.08, auto_min_pad=0.03, edgecolor='#99582a', linewidth=1.3, linestyle=(0, (7, 3.5))):
    start_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)

    x0 = mdates.date2num(start_date)
    x1 = mdates.date2num(end_date)

    if (y_bottom is not None) and (y_top is not None):
        y0 = y_bottom
        y1 = y_top
    else:
        sub = df[(df['date'] >= start_date) & (df['date'] <= end_date)].copy()
        if sub.empty:
            return

        vals = sub[list(line_cols)].values.flatten()
        vals = pd.Series(vals).dropna().values
        if len(vals) == 0:
            return

        y_min_local = vals.min()
        y_max_local = vals.max()
        y_range_local = y_max_local - y_min_local

        pad = max(y_range_local * auto_pad_ratio, auto_min_pad)

        y0 = y_min_local - pad
        y1 = y_max_local + pad

    rect = Rectangle(
        (x0, y0),
        x1 - x0,
        y1 - y0,
        fill=False,
        edgecolor=edgecolor,
        linewidth=linewidth,
        linestyle=linestyle,
        zorder=2
    )
    ax.add_patch(rect)


# ========================
# 3. Single-figure plotting function
# ========================
def plot_phi_with_vertical_line(df, output_dir, line_date):
    # ========================
    # Plotting data time range: 2018-09-06 to 2020-05-06
    # ========================
    plot_start = pd.to_datetime('2018-09-06')
    plot_end = pd.to_datetime('2020-05-06')

    df = df[
        (df['date'] >= plot_start) &
        (df['date'] <= plot_end)
    ].copy()

    if df.empty:
        raise ValueError(
            f"No data are available within the specified time range {plot_start.date()} to {plot_end.date()}. Please check the date column in the Excel file."
        )
    fig, ax = plt.subplots(figsize=(FIG_WIDTH, FIG_HEIGHT))

    # Main curve colors
    color_global = '#1f4e79'
    color_avg    = '#ca6702'
    color_diff   = '#2e8b57'

    # Colors of the three interval boxes
    box1_color = '#ffbe0b'
    box2_color = '#fb5607'
    box3_color = '#8338ec'
    box4_color = '#3a86ff'     # Slightly brighter steel blue
    
    # Curves
    ax.plot(
        df['date'], df['global_phi'],
        color=color_global,
        linewidth=1.55,
        label=r'$\Phi_{\mathrm{global}}$',
        zorder=4
    )

    ax.plot(
        df['date'], df['average_phi'],
        color=color_avg,
        linewidth=1.50,
        label=r'$\overline{\Phi}_{\mathrm{regional}}$',
        zorder=4
    )

    ax.plot(
        df['date'], df['diff_phi'],
        color=color_diff,
        linewidth=1.40,
        label=r'$\Delta \Phi_{\mathrm{rg}}$',
        zorder=4
    )

    # Axis labels
    ax.set_xlabel('Date')
    ax.set_ylabel(r'$\Phi$')

    # X-axis ticks
    major_tick_dates = pd.to_datetime([
        '2018-11',
        '2019-03',
        '2019-07',
        '2019-12',
        '2020-03'
    ])
    ax.xaxis.set_major_locator(FixedLocator(mdates.date2num(major_tick_dates)))
    ax.xaxis.set_minor_locator(mdates.MonthLocator(interval=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

    # Y-axis minor ticks
    ax.yaxis.set_minor_locator(AutoMinorLocator(5))

    # Tick style
    ax.tick_params(axis='both', which='major', direction='in', bottom=True, left=True, top=False, right=False)
    ax.tick_params(axis='both', which='minor', direction='in', bottom=True, left=True, top=False, right=False)

    for label in ax.get_xticklabels():
        label.set_ha('center')

    # Borders
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('black')
        spine.set_linewidth(0.7)

    ax.grid(False)
    ax.margins(x=0.01)

    # X-axis display range only up to 2020-05-01
    ax.set_xlim(plot_start, plot_end)

    # Overall y-axis range
    y_min = min(df['global_phi'].min(), df['average_phi'].min(), df['diff_phi'].min())
    y_max = max(df['global_phi'].max(), df['average_phi'].max(), df['diff_phi'].max())
    y_range = y_max - y_min
    ax.set_ylim(y_min - 0.06 * y_range, y_max + 0.06 * y_range)

    # Vertical line
    ax.axvline(
        pd.to_datetime(line_date),
        color='0.35',
        linestyle='-',
        linewidth=1.1,
        zorder=3
    )

    # Three interval boxes
    add_custom_box(
        ax, df,
        '2019-05-03', '2019-06-30',
        y_bottom=-0.10, y_top=0.97,
        edgecolor=box1_color,
        linewidth=1.3,
        linestyle=(0, (7, 3.5))
    )

    add_custom_box(
        ax, df,
        '2019-07-03', '2019-08-05',
        y_bottom=-0.10, y_top=0.97,
        edgecolor=box2_color,
        linewidth=1.3,
        linestyle=(0, (7, 3.5))
    )

    add_custom_box(
        ax, df,
        '2019-08-09', '2019-10-31',
        y_bottom=-0.10, y_top=0.97,
        edgecolor=box3_color,
        linewidth=1.3,
        linestyle=(0, (7, 3.5))
    )

    add_custom_box(
        ax, df,
        '2019-05-25', '2019-6-18',
        y_bottom=-0.05, y_top=0.5,
        edgecolor=box4_color,
        linewidth=1.3,
        linestyle=(0, (7, 3.5))
    )

    # Legend
    ax.legend(
        loc='lower right',
        bbox_to_anchor=(1.01, 0.42),
        frameon=False,
        handlelength=1,
        handletextpad=0.45,
        borderpad=0.12,
        labelspacing=0.22
    )

    plt.tight_layout(pad=0.45)

    # Save
    os.makedirs(output_dir, exist_ok=True)
    date_str = pd.to_datetime(line_date).strftime('%Y%m%d')

    png_path = os.path.join(output_dir, f'Phi_vertical_{date_str}.png')
    pdf_path = os.path.join(output_dir, f'Phi_vertical_{date_str}.pdf')

    plt.savefig(png_path, dpi=600, bbox_inches='tight')
    plt.savefig(pdf_path, bbox_inches='tight')
    plt.close()

    print(f"Saved: {png_path}")
    print(f"Saved: {pdf_path}")


# ========================
# 4. Batch plotting
# ========================
def batch_plot_all_vertical_lines(df, output_dir, start_date='2019-05-03', end_date='2019-10-31'):
    date_list = pd.date_range(start=start_date, end=end_date, freq='D')

    print(f'A total of {len(date_list)} figures need to be plotted...')
    for i, one_date in enumerate(date_list, start=1):
        print(f'[{i}/{len(date_list)}] Plotting: {one_date.strftime("%Y-%m-%d")}')
        plot_phi_with_vertical_line(df, output_dir, one_date)

    print('All figures have been plotted.')


# ========================
# 5. Main program
# ========================
if __name__ == "__main__":
    excel_path = r"D:\a\master\Earthquake-US\Fig-Use-2\Fig-3\IMS_phi_values_data.xlsx"
    output_dir = r"D:\a\master\Earthquake-US\Fig-Over-Output\Fig-S5\a"

    df_phi = load_phi_data(excel_path)

    batch_plot_all_vertical_lines(
        df=df_phi,
        output_dir=output_dir,
        start_date='2019-05-03',
        end_date='2019-10-31'
    )'''

In [ ]:
# Use the assembled images to create a GIF
'''
import os
import re
from PIL import Image

# =========================
# 1. Input and output paths
# =========================
input_folder = r'D:\a\master\Earthquake-US\Fig-Over-Output\Fig-S5\b'
output_gif = r'D:\a\master\Earthquake-US\Fig-Over-Output\Fig-S5\c\output.gif'

# =========================
# 2. Parameter settings
# =========================
duration = 500   # Display duration of each image, in milliseconds; 500 = 0.5 seconds
loop = 0         # 0 means infinite loop

# Supported image formats
valid_ext = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')

# =========================
# 3. Natural sorting function
#    Sort 1, 2, 10 in the normal numerical order
# =========================
def natural_key(s):
    return [int(text) if text.isdigit() else text.lower()
            for text in re.split(r'(\d+)', s)]

# =========================
# 4. Get image files
# =========================
image_files = [
    f for f in os.listdir(input_folder)
    if f.lower().endswith(valid_ext)
]

image_files.sort(key=natural_key)

if not image_files:
    raise ValueError("No image files found in the folder!")

print("The following images were found:")
for f in image_files:
    print(f)

# =========================
# 5. Read images and unify their sizes
# =========================
images = []
first_image = Image.open(os.path.join(input_folder, image_files[0])).convert("RGB")
base_size = first_image.size
images.append(first_image)

for file in image_files[1:]:
    img_path = os.path.join(input_folder, file)
    img = Image.open(img_path).convert("RGB")
    
    # If the size is different, resize it to match the first image
    if img.size != base_size:
        img = img.resize(base_size, Image.LANCZOS)
    
    images.append(img)

# =========================
# 6. Save as GIF
# =========================
images[0].save(output_gif, save_all=True, append_images=images[1:], duration=duration, loop=loop)

print(f"\nGIF generated: {output_gif}")
'''